# CS513 Phase-II — Data Cleaning Workflow
## Chicago Food Inspections (2010–2025)

**Team79**

| Member | Illinois email | Phase-II role |
|---|---|---|
| Sid Wanjara | swanj2@illinois.edu | Change-summary table (§2a), IC-violation reports (§2b), `queries.txt` |
| Drew Patel | drewp4@illinois.edu | Workflow models W1/W2, Conclusions (§4), report assembly, ZIP/Box |
| Shray Srivastava | ssriv5@illinois.edu | **This notebook** — all data cleaning, provenance, cell-level change counts, Report §1 |

---

### Why this notebook exists

This notebook **is** the inner data-cleaning workflow `W2` and, because we did not use OpenRefine, it
is also our **operation-history / provenance artifact** (it replaces `OpenRefineHistory.json`; a
machine-readable export is written at the end as `OtherToolHistory.json`).

Every high-level step below is a numbered markdown header followed by:

1. **What** the step does,
2. **Why it is needed for U1** (necessary / merely useful — this text feeds Report §1.3),
3. the code, and
4. a **before → after** measurement, so the report's change tables are a copy job rather than a
   re-derivation.

### Use cases

| | Use case | Cleaning verdict |
|---|---|---|
| **U0** | Count inspections by `Results`. | *No cleaning needed* — `Results` is already a clean 7-value controlled vocabulary. Used here as a control: we deliberately do **not** re-bucket it. |
| **U1** | **Identify the most common violation categories associated with failed inspections, broken down by facility type and risk level.** | *Cleaning is necessary and sufficient.* This is the target of the whole workflow. |
| **U2** | Determine whether a specific establishment *caused* a foodborne-illness outbreak. | *No amount of cleaning suffices* — `D` contains no illness, patient, or lab-confirmation records. |

**U1 decomposed into the fields it touches** — this is the scope test applied to every step below:

```
   most common violation categories   ->  Violations      (semi-structured, multi-valued free text)
   associated with failed inspections ->  Results         (already clean)
   by facility type                   ->  Facility Type   (520 distinct values, 5,264 blanks)
   by risk level                      ->  Risk            (blanks + an undocumented 'All' category)
   [scope check: is this Chicago?]    ->  City / State    (spelling variants, blanks)
   [entity key]                       ->  Inspection ID   (must remain a primary key)
```

### Tooling, and the deviation from the Phase-I plan

| | Phase-I plan | Phase-II actual | Why the change |
|---|---|---|---|
| Cleaning tool | OpenRefine (owner: Drew) | **Python 3 / pandas** (owner: Shray) | The critical-path step is *parsing the multi-valued `Violations` column into a violation-level relation*. OpenRefine can split multi-valued cells, but the per-chunk regex parse (`N. TITLE - Comments: ...`), the date-dependent violation code-book logic, and the 1→N fan-out into a **second output table** are far cleaner as code. Python also gives re-runnable, diff-able provenance. |
| IC checking | SQL (engine unspecified) | **DuckDB SQL, run in-process over the dataframes** | Satisfies the "SQL or Datalog" requirement for `queries.txt` with zero database installation or ETL; the same SQL text runs against the raw table and the cleaned tables, so before/after numbers are strictly comparable. |
| Inner workflow diagram | OR2YW | Hand-built from this notebook's step structure | OR2YW consumes an OpenRefine history JSON, which no longer exists. The numbered step headers here are the node list used for `W2`. |
| Ownership | S3 cleaning = Drew | S3 cleaning = Shray | Re-assigned during Phase-II planning; recorded in Report §1.1 and §4.2. |

### Outputs produced by this notebook

| File | Contents |
|---|---|
| `output/clean_inspections.csv` | One row per inspection (PK `inspection_id`): standardized fields, `facility_group`, quality flags, raw values retained as `*_raw`. |
| `output/clean_violations.csv` | One row per individual violation per inspection — the 1NF decomposition of `Violations`. |
| `output/change_summary.csv` | Per-column cells-changed / cells-filled counts (Report §2a). |
| `output/ic_report.csv` | Every integrity constraint, before-count vs after-count (Report §2b). |
| `output/u1_result.csv` | The answer to U1, computed on `D'`. |
| `output/OtherToolHistory.json` | Machine-readable operation history — our OpenRefine-history substitute. |
| `queries.txt` | All profiling / IC / U1 SQL, exactly as executed. |

---
# Step 0 — Environment, configuration, and the provenance log

**What.** Imports, paths, and a small operation-logging helper.

**Why for U1.** Not a cleaning step. It exists so that every subsequent transformation records
*what changed and by how much* — this is what turns a notebook into an auditable operation history
(Report §3 and the supplementary `OtherToolHistory.json`).

In [1]:
import json, os, re, sys, time, datetime, textwrap
import pandas as pd
import duckdb

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

PROJECT_DIR = os.getcwd()
RAW_CSV     = os.path.join(PROJECT_DIR, "Food-Inspections-20251023.csv")
OUT_DIR     = os.path.join(PROJECT_DIR, "output")
os.makedirs(OUT_DIR, exist_ok=True)

RUN_STARTED = datetime.datetime.now().isoformat(timespec="seconds")

print("python  :", sys.version.split()[0])
print("pandas  :", pd.__version__)
print("duckdb  :", duckdb.__version__)
print("raw csv :", RAW_CSV, "(%.1f MB)" % (os.path.getsize(RAW_CSV)/1e6))
print("outputs :", OUT_DIR)
print("run     :", RUN_STARTED)

python  : 3.9.6
pandas  : 2.3.3
duckdb  : 1.4.5
raw csv : /Users/shraysrivastava/Desktop/cs513/Food-Inspections-20251023.csv (327.9 MB)
outputs : /Users/shraysrivastava/Desktop/cs513/output
run     : 2026-07-27T22:10:10


In [2]:
# ---------------------------------------------------------------- provenance log
# Every transformation appends one record here. It is exported at the end as OtherToolHistory.json,
# which is this project's stand-in for OpenRefineHistory.json.
OPS = []

def log_op(step, op, description, rationale, params=None, before=None, after=None,
           rows_affected=None, columns_in=None, columns_out=None):
    rec = {
        "op_id": "op%03d" % (len(OPS) + 1),
        "step": step,
        "operation": op,
        "description": description,
        "rationale_for_U1": rationale,
        "tool": "python/pandas",
        "params": params or {},
        "columns_in": columns_in or [],
        "columns_out": columns_out or [],
        "before": before,
        "after": after,
        "rows_affected": rows_affected,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    OPS.append(rec)
    detail = []
    if before is not None or after is not None:
        detail.append("before=%s -> after=%s" % (before, after))
    if rows_affected is not None:
        detail.append("rows_affected=%s" % format(rows_affected, ","))
    print("[%s] %s :: %s" % (rec["op_id"], step, op) + ("  |  " + "; ".join(detail) if detail else ""))

NA_SENTINEL = "\x00__NA__\x00"

def changed_count(before_series, after_series):
    # number of cells whose value actually changed (NaN-safe: NaN vs NaN counts as unchanged)
    b = before_series.fillna(NA_SENTINEL).astype(str)
    a = after_series.fillna(NA_SENTINEL).astype(str)
    return int((b != a).sum())

def filled_count(before_series, after_series):
    # number of cells that were blank/NaN before and carry a value after
    return int((before_series.isna() & after_series.notna()).sum())

---
# Step 1 — Load and profile the raw dataset `D`

**What.** Load the raw CSV with **every column as text and no type inference**
(`dtype=str`, `keep_default_na=False`, `na_values=[""]`), then profile it: shape, missingness,
cardinality, value distributions, key uniqueness, and the structure of the `Violations` column.

**Why for U1.** *Necessary*, for two reasons:

1. **Type inference would silently corrupt the data.** `Zip` and `License #` are identifiers, not
   numbers — pandas would turn `"60632"` into `60632`, and zero-padded licence numbers would lose
   their padding. Loading everything as text guarantees that the only changes to `D` are the ones we
   make on purpose and can count.
2. **Profiling defines the cleaning targets.** Every rule below (which city spellings to fold, which
   risk values are out of domain, which facility strings need grouping, the violation regex) is
   derived from the distributions measured *here*, not from assumptions carried over from Phase-I.

We also **re-derive the Phase-I missing-value counts** from the actual file instead of trusting the
Phase-I report, since the city's portal data can drift between downloads.

In [3]:
t0 = time.time()
raw = pd.read_csv(RAW_CSV, dtype=str, keep_default_na=False, na_values=[""])
print("loaded in %.1fs" % (time.time() - t0))
print("shape :", raw.shape)
print("memory: %.2f GB" % (raw.memory_usage(deep=True).sum() / 1e9))
raw.head(3)

loaded in 1.7s
shape : (298869, 17)


memory: 0.61 GB


,Inspection ID,DBA Name,AKA Name,License #,Facility Type,Risk,Address,City,State,Zip,Inspection Date,Inspection Type,Results,Violations,Latitude,Longitude,Location
0,2625743,XOCOBERRY,XOCOBERRY,3046764,Restaurant,Risk 1 (High),5158 S KEDZIE AVE,CHICAGO,IL,60632,10/22/2025,License,Pass,"53. TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED - Comments: OBSERVED ...",41.79908205645,-87.70386302302,"(41.79908205644847, -87.70386302302413)"
1,2625757,"MARZEYA BAKERY J.A.S., INC.","MARZEYA BAKERY J.A.S., INC.",2712762,Restaurant,Risk 2 (Medium),8908 S COMMERCIAL AVE,CHICAGO,IL,60617,10/22/2025,Canvass Re-Inspection,Fail,10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: HANDSINK I...,41.73341000506,-87.55148453485,"(41.733410005055205, -87.55148453485408)"
2,2625731,PATHWAYS TO LEARNING CHILD CARE INC.,PATHWAYS TO LEARNING CHILD CARE INC.,2215782,Daycare Above and Under 2 Years,Risk 1 (High),6535 S KEDZIE AVE,CHICAGO,IL,60629,10/22/2025,License Re-Inspection,Pass,41. WIPING CLOTHS: PROPERLY USED & STORED - Comments: 3-304.14 OBSERVED WET WIPING CLO...,41.77431077413,-87.7028825033,"(41.77431077413166, -87.70288250330134)"


In [4]:
# ------------------------------------------------------------------ column profile of D
prof = pd.DataFrame({
    "n_rows":      len(raw),
    "n_missing":   raw.isna().sum(),
    "pct_missing": (raw.isna().sum() / len(raw) * 100).round(2),
    "n_distinct":  raw.nunique(dropna=True),
    "example":     [raw[c].dropna().iloc[0][:55] if raw[c].notna().any() else "" for c in raw.columns],
})
prof.index.name = "column"
prof

,n_rows,n_missing,pct_missing,n_distinct,example
column,,,,,
Inspection ID,298869,0,0.00,298869,2625743
DBA Name,298869,0,0.00,34146,XOCOBERRY
AKA Name,298869,2412,0.81,32505,XOCOBERRY
License #,298869,18,0.01,47562,3046764
Facility Type,298869,5264,1.76,520,Restaurant
Risk,298869,87,0.03,4,Risk 1 (High)
Address,298869,3,0.00,20207,5158 S KEDZIE AVE
City,298869,162,0.05,89,CHICAGO
State,298869,58,0.02,6,IL


In [5]:
# ------------------------------------------------------------------ Phase-I numbers, re-derived
phase1_reported = {
    "Violations": 83344, "Facility Type": 5264, "AKA Name": 2412, "Latitude": 1022,
    "Longitude": 1022, "Location": 1022, "City": 162, "Risk": 87, "State": 58,
    "Zip": 42, "License #": 18, "Address": 3, "Inspection Type": 1,
}
check = pd.DataFrame({
    "phase1_reported": pd.Series(phase1_reported),
    "phase2_actual":   raw.isna().sum(),
}).dropna(subset=["phase1_reported"])
check["phase1_reported"] = check["phase1_reported"].astype(int)
check["match"] = check.phase1_reported == check.phase2_actual
print(check.to_string())
print("\nAll Phase-I missing-value counts reproduced exactly:", bool(check["match"].all()))

log_op("1. Profile", "verify_phase1_profile",
       "Re-derived the Phase-I missing-value profile from the actual CSV.",
       "Confirms the Phase-I inventory of DQ problems still describes the file we are cleaning.",
       before={"phase1_reported_missing_total": int(check.phase1_reported.sum())},
       after={"phase2_actual_missing_total": int(check.phase2_actual.sum())})

                 phase1_reported  phase2_actual  match
AKA Name                    2412           2412   True
Address                        3              3   True
City                         162            162   True
Facility Type               5264           5264   True
Inspection Type                1              1   True
Latitude                    1022           1022   True
License #                     18             18   True
Location                    1022           1022   True
Longitude                   1022           1022   True
Risk                          87             87   True
State                         58             58   True
Violations                 83344          83344   True
Zip                           42             42   True

All Phase-I missing-value counts reproduced exactly: True
[op001] 1. Profile :: verify_phase1_profile  |  before={'phase1_reported_missing_total': 94457} -> after={'phase2_actual_missing_total': 94457}


In [6]:
# ------------------------------------------------------------------ key + distribution profile
print("PRIMARY KEY  Inspection ID")
print("  rows            :", format(len(raw), ","))
print("  distinct ids    :", format(raw["Inspection ID"].nunique(), ","))
print("  duplicate ids   :", int(raw["Inspection ID"].duplicated().sum()))
print("  all ids numeric :", bool(raw["Inspection ID"].str.fullmatch(r"\d+").all()))

d_probe = pd.to_datetime(raw["Inspection Date"], format="%m/%d/%Y", errors="coerce")
print("\nInspection Date")
print("  unparseable     :", int(d_probe.isna().sum()))
print("  range           :", d_probe.min().date(), "->", d_probe.max().date())

for c in ["Results", "Risk", "State"]:
    print("\n%s  (%d distinct)" % (c, raw[c].nunique()))
    print(raw[c].value_counts(dropna=False).to_string())

PRIMARY KEY  Inspection ID
  rows            : 298,869
  distinct ids    : 298,869
  duplicate ids   : 0
  all ids numeric : True

Inspection Date
  unparseable     : 0
  range           : 2010-01-04 -> 2025-10-22

Results  (7 distinct)
Results
Pass                    154452
Fail                     57819
Pass w/ Conditions       44716
Out of Business          24726
No Entry                 12953
Not Ready                 4110
Business Not Located        93

Risk  (4 distinct)
Risk
Risk 1 (High)      221383
Risk 2 (Medium)     53822
Risk 3 (Low)        23497
NaN                    87
All                    80

State  (6 distinct)
State
IL     298793
NaN        58
IN         11
CA          3
WI          2
CO          1
NY          1


In [7]:
print("City  (%d distinct raw values)  -- top 25" % raw["City"].nunique())
print(raw["City"].value_counts(dropna=False).head(25).to_string())
print("\nFacility Type  (%d distinct raw values)  -- top 20" % raw["Facility Type"].nunique())
print(raw["Facility Type"].value_counts(dropna=False).head(20).to_string())
print("\nInspection Type  (%d distinct raw values)  -- top 12" % raw["Inspection Type"].nunique())
print(raw["Inspection Type"].value_counts(dropna=False).head(12).to_string())

City  (89 distinct raw values)  -- top 25
City
CHICAGO              297676
Chicago                 483
NaN                     162
chicago                 158
CCHICAGO                 61
SCHAUMBURG               28
EVANSTON                 24
CHicago                  22
MAYWOOD                  15
ELK GROVE VILLAGE        13
CHICAGOCHICAGO           12
OAK PARK                 11
CICERO                    9
CHICAGOO                  9
SKOKIE                    9
BERWYN                    8
INACTIVE                  8
CHICAGO.                  8
NILES NILES               7
MERRIVILLE                7
CALUMET CITY              7
ELMHURST                  7
312CHICAGO                6
CHCHICAGO                 6
PLAINFIELD                5

Facility Type  (520 distinct raw values)  -- top 20
Facility Type
Restaurant                         201947
Grocery Store                       36328
School                              18822
Children's Services Facility         7111
NaN               

In [8]:
# ------------------------------------------------------------------ structure of the Violations column
v_col = raw["Violations"]
pipes = v_col.fillna("").str.count(r"\|")
n_facts_in_text = int((pipes[v_col.notna()] + 1).sum())

print("Violations column")
print("  rows with no violation text    :", format(int(v_col.isna().sum()), ","))
print("  rows with violation text       :", format(int(v_col.notna().sum()), ","))
print("  ... of which multi-valued      :", format(int((pipes > 0).sum()), ","),
      " <- a first-normal-form violation")
print("  max violations packed in 1 cell:", int(pipes.max() + 1))
print("  violation facts trapped in text:", format(n_facts_in_text, ","))
print("\nExample cell:\n")
print(textwrap.fill(v_col.dropna().iloc[1][:700], 116))

print("\n\nResults x has-violation-text crosstab (motivates the fail_missing_violations flag):")
print(pd.crosstab(raw["Results"], v_col.notna().rename("has_violation_text")).to_string())

log_op("1. Profile", "profile_violations_column",
       "Measured the multi-valuedness of the Violations column.",
       "U1 counts violation categories; they are unusable while packed into one free-text cell.",
       before={"violation_facts_in_free_text": n_facts_in_text, "atomic_violation_rows": 0})

Violations column
  rows with no violation text    : 83,344
  rows with violation text       : 215,525
  ... of which multi-valued      : 186,666  <- a first-normal-form violation
  max violations packed in 1 cell: 40
  violation facts trapped in text: 974,597

Example cell:

10. ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE - Comments: HANDSINK IN REAR PREP AREA IN POOR
REPAIR AT THIS TIME. SINK IS VERY SLOW DRAINING. INSTRUCTED MANAGER TO USE HANDSINK AT FRONT PREP AREA 15 FEET AWAY.
CORRECT AND MAINTAIN  PRIORITY FOUNDATION 7-38-030(C) CITATION ISSUED | 40. PERSONAL CLEANLINESS - Comments: FOOD
HANDLER IN NEED OF EFFECTIVE HAIR RESTRAINT. INSTRUCTED TO CORRECT AND MAINTAIN | 47. FOOD & NON-FOOD CONTACT
SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED - Comments: WORN TORN GASKETS IN NEED OF REPLACING ON
FRONT REACH IN COOLER. REAR REACH IN COOLER GASKETS IN NEED OF CLEANING TO REMOVE ALL BUILD UP. INSTRUCTED TO DETAIL
CLEAN AND MAINTAIN


Results x has-violati

---
# Step 2 — Standardize the categorical and scope fields

Raw values are **never overwritten**. Each cleaned field becomes a new column and the original is
kept beside it as `*_raw`, so every cell change stays auditable and reversible (Step 8 counts them
by diffing the two).

## Step 2.1 — `City`

**What.** Upper-case, strip punctuation and repeated whitespace, collapse doubled tokens
(`CHICAGOCHICAGO`, `NILES NILES`), apply a short list of explicit corrections, then fold anything
within **Levenshtein distance ≤ 2 of `CHICAGO`** into `CHICAGO`. Placeholder junk (`INACTIVE`,
`CHARLES A HAYES`) and blanks become `UNKNOWN`.

**Why for U1.** *Necessary but indirect.* U1 never groups by city — but it is stated over
**Chicago** inspections, so we must be able to verify the scope ("is this really one city's data,
and how much of it isn't?"). With 90 raw spellings that question cannot be answered; after cleaning
it is a single count. City is also the clearest illustration of the difference between
*inconsistent representation* and *genuinely different values*.

**Deliberate non-action, documented for the report.** We do **not** fuzzy-merge suburb names.
Frequency-based clustering would map the *correctly* spelled `MERRILLVILLE` (1 row) into the
*misspelled* `MERRIVILLE` (7 rows), and `OLYMPIA FIELDS` / `OOLYMPIA FIELDS` carries the same hazard.
Those ~50 non-Chicago rows are outside U1's scope, so the risk of a wrong merge outweighs the gain;
they are normalized for case and punctuation only.

In [9]:
def levenshtein(a, b):
    if a == b: return 0
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

CITY_PLACEHOLDERS = {"INACTIVE", "CHARLES A HAYES"}         # not city names at all
CITY_EXPLICIT     = {"CH": "CHICAGO",                        # unambiguous abbreviation (IL, 606xx zip)
                     "CHICAGOBEDFORD PARK": "BEDFORD PARK"}  # two city names concatenated

def _norm_text(s):
    # upper-case, drop anything that is not a letter or a space, squeeze whitespace
    return re.sub(r"\s+", " ", re.sub(r"[^A-Z ]", " ", str(s).upper())).strip()

def _collapse_repeat(u):
    # CHICAGOCHICAGO -> CHICAGO ; NILES NILES -> NILES
    toks = u.split(" ")
    if len(toks) == 2 and toks[0] == toks[1]:
        return toks[0]
    for k in (2, 3, 4):
        if len(u) % k == 0 and u[: len(u) // k] * k == u:
            return u[: len(u) // k]
    return u

def clean_city(s):
    # returns (city_clean, rule_id)
    if s is None or isinstance(s, float) or str(s).strip() == "":
        return "UNKNOWN", "C0_blank_to_UNKNOWN"
    u = _norm_text(s)
    if u in CITY_PLACEHOLDERS or u == "":
        return "UNKNOWN", "C1_placeholder_to_UNKNOWN"
    if u in CITY_EXPLICIT:
        return CITY_EXPLICIT[u], "C2_explicit_correction"
    c = _collapse_repeat(u)
    if c != u:
        return c, "C3_collapse_repeated_token"
    if c != "CHICAGO" and levenshtein(c, "CHICAGO") <= 2:
        return "CHICAGO", "C4_fuzzy_merge_to_CHICAGO"
    return c, ("C5_case_punctuation_only" if c != str(s) else "C6_unchanged")

_res       = raw["City"].map(clean_city)
city_clean = _res.map(lambda t: t[0])
city_rule  = _res.map(lambda t: t[1])

print("distinct City values : %d  ->  %d" % (raw["City"].nunique(dropna=False), city_clean.nunique()))
print("rows equal to CHICAGO: %s  ->  %s" % (format(int((raw["City"] == "CHICAGO").sum()), ","),
                                             format(int((city_clean == "CHICAGO").sum()), ",")))
print("\nrule usage:")
print(city_rule.value_counts().to_string())
print("\nevery raw value the rules actually rewrote:")
_chg = pd.DataFrame({"raw": raw["City"], "clean": city_clean, "rule": city_rule})
_chg = _chg[_chg.raw.fillna("<BLANK>") != _chg.clean]
print(_chg.groupby(["rule", "raw", "clean"], dropna=False).size()
      .rename("n_rows").reset_index().to_string(index=False))

distinct City values : 90  ->  73
rows equal to CHICAGO: 297,676  ->  298,450

rule usage:
City
C6_unchanged                  297910
C5_case_punctuation_only         680
C0_blank_to_UNKNOWN              162
C4_fuzzy_merge_to_CHICAGO         84
C3_collapse_repeated_token        19
C1_placeholder_to_UNKNOWN         12
C2_explicit_correction             2

every raw value the rules actually rewrote:
                      rule                 raw        clean  n_rows
       C0_blank_to_UNKNOWN                 NaN      UNKNOWN     162
 C1_placeholder_to_UNKNOWN     CHARLES A HAYES      UNKNOWN       4
 C1_placeholder_to_UNKNOWN            INACTIVE      UNKNOWN       8
    C2_explicit_correction                  CH      CHICAGO       1
    C2_explicit_correction chicagoBEDFORD PARK BEDFORD PARK       1
C3_collapse_repeated_token      CHICAGOCHICAGO      CHICAGO      12
C3_collapse_repeated_token         NILES NILES        NILES       7
 C4_fuzzy_merge_to_CHICAGO            CCHICAGO      CHIC

In [10]:
print("non-CHICAGO values surviving cleaning (%d rows, %.3f%% of D):"
      % (int((city_clean != "CHICAGO").sum()), (city_clean != "CHICAGO").mean() * 100))
print(city_clean[city_clean != "CHICAGO"].value_counts().to_string())

log_op("2.1 City", "normalize_and_cluster_city",
       "Upper/strip/de-punctuate, collapse repeated tokens, apply explicit corrections, then "
       "Levenshtein<=2 merge of Chicago spelling variants; blanks and placeholders -> UNKNOWN.",
       "U1 is scoped to Chicago inspections; that scope can only be verified once the 90 spellings "
       "of the city collapse to canonical values. Suburb-level fuzzy merging deliberately NOT done.",
       params={"distance_metric": "levenshtein", "threshold": 2, "anchor": "CHICAGO",
               "explicit_map": CITY_EXPLICIT, "placeholders": sorted(CITY_PLACEHOLDERS)},
       before={"distinct": int(raw["City"].nunique(dropna=False)),
               "exact_CHICAGO": int((raw["City"] == "CHICAGO").sum()),
               "missing": int(raw["City"].isna().sum())},
       after={"distinct": int(city_clean.nunique()),
              "exact_CHICAGO": int((city_clean == "CHICAGO").sum()), "missing": 0},
       rows_affected=changed_count(raw["City"], city_clean),
       columns_in=["City"], columns_out=["city_clean"])

non-CHICAGO values surviving cleaning (419 rows, 0.140% of D):
City
UNKNOWN                 174
SCHAUMBURG               28
EVANSTON                 24
MAYWOOD                  16
ELK GROVE VILLAGE        13
OAK PARK                 11
SKOKIE                    9
CICERO                    9
BERWYN                    8
NILES                     7
CALUMET CITY              7
ELMHURST                  7
MERRIVILLE                7
WORTH                     5
PLAINFIELD                5
SUMMIT                    5
NAPERVILLE                5
ALSIP                     4
BRIDGEVIEW                4
HIGHLAND PARK             4
EAST HAZEL CREST          3
ROSEMONT                  3
SCHILLER PARK             3
BURNHAM                   2
BURBANK                   2
BANNOCKBURNDEERFIELD      2
CHICAGO HEIGHTS           2
STREAMWOOD                2
OAK LAWN                  2
EVERGREEN PARK            2
BLUE ISLAND               2
LAKE ZURICH               2
OOLYMPIA FIELDS           1
FRANKFOR

## Step 2.2 — `State`

**What.** Upper-case/trim; blank → `UNKNOWN`; keep the 18 genuinely out-of-state rows but raise an
`out_of_state` flag.

**Why for U1.** *Useful rather than strictly necessary* — the same scope-verification role as
`City`. We keep the non-`IL` rows instead of deleting them: whether to exclude them is an
**analysis** decision, and making it a flag keeps `|D'| = |D|` and the decision reversible
(scope guardrail: *never delete rows to fix a problem*).

In [11]:
state_clean  = raw["State"].fillna("").str.strip().str.upper().replace("", "UNKNOWN")
out_of_state = ~state_clean.isin(["IL", "UNKNOWN"])

print("distinct State : %d -> %d" % (raw["State"].nunique(dropna=False), state_clean.nunique()))
print(state_clean.value_counts().to_string())
print("\nrows flagged out_of_state :", int(out_of_state.sum()))
print("rows with missing state   :", int((state_clean == "UNKNOWN").sum()))

log_op("2.2 State", "normalize_state",
       "Upper/trim State; blank -> 'UNKNOWN'; non-IL rows flagged rather than deleted.",
       "Scope check for U1 (Chicago, IL). Flag-not-delete keeps |D'| = |D| and the choice reversible.",
       before={"distinct": int(raw["State"].nunique(dropna=False)),
               "missing": int(raw["State"].isna().sum())},
       after={"distinct": int(state_clean.nunique()), "missing": 0,
              "flagged_out_of_state": int(out_of_state.sum())},
       rows_affected=changed_count(raw["State"], state_clean),
       columns_in=["State"], columns_out=["state_clean", "out_of_state"])

distinct State : 7 -> 7
State
IL         298793
UNKNOWN        58
IN             11
CA              3
WI              2
CO              1
NY              1

rows flagged out_of_state : 18


rows with missing state   : 58
[op004] 2.2 State :: normalize_state  |  before={'distinct': 7, 'missing': 58} -> after={'distinct': 7, 'missing': 0, 'flagged_out_of_state': 18}; rows_affected=58


## Step 2.3 — `Risk`

**What.** Map onto exactly four values — `Risk 1 (High)`, `Risk 2 (Medium)`, `Risk 3 (Low)`,
`Unknown`. Blanks (87 rows) → `Unknown`; the undocumented `All` category (80 rows) → `Unknown`. Also
derive a compact ordinal `risk_level` ∈ {1, 2, 3, NULL}.

**Why for U1.** ***Necessary — U1 groups by risk level.*** Any value outside the three-level ordinal
domain either breaks the grouping or invents a phantom fourth risk tier in the U1 result.

**Open decision, flagged for the report.** `All` is not defined in the City of Chicago data
dictionary and is not an ordinal level; treating it as a fourth "risk" would be worse than treating
it as unknown, and it is 80 rows (0.03% of `D`). We therefore fold it into `Unknown` **and report it
as an assumption**, not a fact. `risk_is_imputed` marks those rows so a reviewer can isolate them.

In [12]:
RISK_MAP = {
    "RISK 1 (HIGH)": "Risk 1 (High)", "RISK 2 (MEDIUM)": "Risk 2 (Medium)",
    "RISK 3 (LOW)": "Risk 3 (Low)", "ALL": "Unknown", "": "Unknown",
}
_risk_key       = raw["Risk"].fillna("").str.strip().str.upper().str.replace(r"\s+", " ", regex=True)
risk_clean      = _risk_key.map(RISK_MAP).fillna("Unknown")
risk_level      = pd.to_numeric(risk_clean.str.extract(r"Risk (\d)")[0], errors="coerce").astype("Int64")
risk_is_imputed = _risk_key.isin(["", "ALL"])

print("Risk before:"); print(raw["Risk"].value_counts(dropna=False).to_string())
print("\nRisk after:");  print(risk_clean.value_counts(dropna=False).to_string())
print("\nrows where risk was blank or 'All' (risk_is_imputed):", int(risk_is_imputed.sum()))

log_op("2.3 Risk", "map_risk_to_ordinal_domain",
       "Mapped Risk onto {Risk 1 (High), Risk 2 (Medium), Risk 3 (Low), Unknown}; blank (87) and the "
       "undocumented 'All' (80) -> Unknown, marked with risk_is_imputed.",
       "NECESSARY: U1 breaks results down by risk level, so risk must be a closed ordinal domain.",
       params={"map": RISK_MAP,
               "assumption": "'All' is not an ordinal risk level -> treated as Unknown"},
       before={"distinct": int(raw["Risk"].nunique(dropna=False)),
               "out_of_domain_rows": int(_risk_key.isin(["", "ALL"]).sum())},
       after={"distinct": int(risk_clean.nunique()), "out_of_domain_rows": 0},
       rows_affected=changed_count(raw["Risk"], risk_clean),
       columns_in=["Risk"], columns_out=["risk_clean", "risk_level", "risk_is_imputed"])

Risk before:
Risk
Risk 1 (High)      221383
Risk 2 (Medium)     53822
Risk 3 (Low)        23497
NaN                    87
All                    80

Risk after:
Risk
Risk 1 (High)      221383
Risk 2 (Medium)     53822
Risk 3 (Low)        23497
Unknown               167

rows where risk was blank or 'All' (risk_is_imputed): 167
[op005] 2.3 Risk :: map_risk_to_ordinal_domain  |  before={'distinct': 5, 'out_of_domain_rows': 167} -> after={'distinct': 4, 'out_of_domain_rows': 0}; rows_affected=167


## Step 2.4 — `Results` (the U0 control)

**What.** Trim whitespace only, then *verify* the value set against the 7 documented outcomes. No
re-bucketing, no case folding beyond trimming.

**Why for U1.** *Necessary to check, necessary to leave alone.* U1 filters on `Results = 'Fail'`, so
the column must be trustworthy — and profiling shows it already is. This is exactly our **U0**
("no cleaning needed") case, and the honest engineering answer is to **assert** the constraint and
change nothing. Deliberately doing nothing, with evidence, is a result worth reporting.

In [13]:
RESULTS_DOMAIN = {"Pass", "Fail", "Pass w/ Conditions", "Out of Business",
                  "No Entry", "Not Ready", "Business Not Located"}
result_clean = raw["Results"].str.strip()
unexpected   = set(result_clean.dropna().unique()) - RESULTS_DOMAIN

print("distinct Results:", result_clean.nunique(), "| unexpected values:", unexpected or "none")
print(result_clean.value_counts().to_string())
print("\ncells changed by trimming:", changed_count(raw["Results"], result_clean),
      " <- U0: the column was already clean")

log_op("2.4 Results", "verify_results_domain",
       "Trimmed whitespace and asserted Results against the 7 documented outcomes. No re-bucketing.",
       "U1 filters on Results='Fail'. Profiling shows the column is already a clean controlled "
       "vocabulary (our U0 case), so the correct action is to verify, not to transform.",
       before={"distinct": int(raw["Results"].nunique()), "out_of_domain": len(unexpected)},
       after={"distinct": int(result_clean.nunique()), "out_of_domain": 0},
       rows_affected=changed_count(raw["Results"], result_clean),
       columns_in=["Results"], columns_out=["result"])

distinct Results: 7 | unexpected values: none
Results
Pass                    154452
Fail                     57819
Pass w/ Conditions       44716
Out of Business          24726
No Entry                 12953
Not Ready                 4110
Business Not Located        93

cells changed by trimming: 0  <- U0: the column was already clean
[op006] 2.4 Results :: verify_results_domain  |  before={'distinct': 7, 'out_of_domain': 0} -> after={'distinct': 7, 'out_of_domain': 0}; rows_affected=0


## Step 2.5 — `Inspection Type` and `Inspection Date`

**What.** `Inspection Type`: squeeze whitespace and apply consistent Title Case so that
`OUT OF BUSINESS` / `Out of Business` / `out of business` stop being three categories; derive an
`is_reinspection` flag. `Inspection Date`: parse `MM/DD/YYYY` → ISO `YYYY-MM-DD` and derive
`inspection_year` and `code_era` — see Step 4, where the violation code book turns out to change on
**2018-07-01**, which makes the date semantically load-bearing.

**Why for U1.** `Inspection Type` is *useful, not necessary* — U1 does not group by it, but it is
the first field an analyst reaches for afterwards ("were re-inspections worse?"), and folding pure
case variants is zero-risk. `Inspection Date` is ***necessary***: without a parsed date we cannot
tell which violation code book a row's numbers belong to, and U1's categories would silently merge
two different code systems (proved in Step 4).

In [14]:
SMALL_WORDS = {"of", "and", "for", "to", "the", "a", "in", "w/", "on"}

def title_case(s):
    if s is None or isinstance(s, float):
        return s
    s = re.sub(r"\s+", " ", str(s)).strip()
    parts = []
    for i, w in enumerate(s.split(" ")):
        lw = w.lower()
        if i > 0 and lw in SMALL_WORDS:
            parts.append(lw)
        elif re.search(r"\d", w) or (w.isupper() and w.isalpha() and len(w) <= 3):
            parts.append(w.upper())
        else:
            parts.append(lw[:1].upper() + lw[1:])
    return " ".join(parts)

inspection_type = raw["Inspection Type"].map(title_case)
is_reinspection = inspection_type.fillna("").str.contains(r"re-?inspection", case=False, regex=True)

inspection_date  = pd.to_datetime(raw["Inspection Date"], format="%m/%d/%Y", errors="coerce")
inspection_year  = inspection_date.dt.year.astype("Int64")
CODE_CHANGE_DATE = pd.Timestamp("2018-07-01")
code_era         = (inspection_date >= CODE_CHANGE_DATE).map({True: "2018_code", False: "pre_2018_code"})

print("Inspection Type distinct: %d -> %d" % (raw["Inspection Type"].nunique(dropna=False),
                                              inspection_type.nunique(dropna=False)))
print("\nexamples of merged case variants:")
_m = pd.DataFrame({"raw": raw["Inspection Type"], "clean": inspection_type})
_m = _m[_m.raw.fillna("") != _m.clean.fillna("")]
print(_m.groupby(["clean", "raw"]).size().rename("n").reset_index().head(20).to_string(index=False))
print("\nis_reinspection rows:", int(is_reinspection.sum()))
print("\ndates unparseable:", int(inspection_date.isna().sum()),
      "| range:", inspection_date.min().date(), "->", inspection_date.max().date())
print("code era split:"); print(code_era.value_counts().to_string())

log_op("2.5 Inspection Type", "normalize_inspection_type",
       "Whitespace squeeze plus consistent Title Case; derived is_reinspection.",
       "USEFUL (not required by U1): removes pure case-variant duplicates so later slices of the U1 "
       "result by inspection type are not fragmented.",
       before={"distinct": int(raw["Inspection Type"].nunique(dropna=False))},
       after={"distinct": int(inspection_type.nunique(dropna=False))},
       rows_affected=changed_count(raw["Inspection Type"], inspection_type),
       columns_in=["Inspection Type"], columns_out=["inspection_type", "is_reinspection"])

log_op("2.5 Inspection Date", "parse_dates_and_derive_code_era",
       "Parsed MM/DD/YYYY -> ISO date; derived inspection_year and code_era (split at 2018-07-01).",
       "NECESSARY: the violation code book changed on 2018-07-01, so the same violation NUMBER means "
       "different things before and after. Without the parsed date U1 would merge two code systems.",
       params={"input_format": "%m/%d/%Y", "code_change_date": "2018-07-01"},
       before={"unparseable": int(inspection_date.isna().sum()), "format": "MM/DD/YYYY text"},
       after={"unparseable": int(inspection_date.isna().sum()), "format": "ISO-8601 date"},
       rows_affected=len(raw),
       columns_in=["Inspection Date"],
       columns_out=["inspection_date", "inspection_year", "code_era"])

Inspection Type distinct: 111 -> 101

examples of merged case variants:
                            clean                               raw     n
        1315 License Reinspection         1315 license reinspection     1
                         Addendum                          ADDENDUM     1
                       Assessment                        ASSESSMENT     1
                           Canvas                            CANVAS     1
                          Canvass                           CANVASS     1
Canvass RE Inspection of Close UP CANVASS RE INSPECTION OF CLOSE UP     1
            Canvass Re-inspection             Canvass Re-Inspection 33511
     Canvass School/special Event      CANVASS SCHOOL/SPECIAL EVENT     1
           Canvass Special Events            CANVASS SPECIAL EVENTS     1
             Canvass for RIB Fest              CANVASS FOR RIB FEST     1
            Canvass/special Event             CANVASS/SPECIAL EVENT     1
               Changed Court Date       

## Step 2.6 — Identity, address and geo fields (trim only)

**What.** `DBA Name`, `AKA Name`, `Address`: whitespace squeeze and upper-case. `License #`, `Zip`:
trim. `Latitude`/`Longitude`: cast to float, blanks preserved.

**Why for U1.** *Useful, deliberately minimal.* U1 does not group by establishment name or address,
so entity resolution over `DBA Name` (a project in its own right — chains, franchise numbering,
punctuation) is **out of scope** under the guardrail *"don't over-clean fields U1 doesn't touch"*.
We normalize only what is free: whitespace and case. We do **not** invent missing zips or
coordinates — fabricating a location to satisfy a completeness metric would be a data-quality
*regression*, so they stay null and get flagged instead.

In [15]:
def squeeze_upper(s):
    if s is None or isinstance(s, float):
        return s
    return re.sub(r"\s+", " ", str(s)).strip().upper()

dba_name    = raw["DBA Name"].map(squeeze_upper)
aka_name    = raw["AKA Name"].map(squeeze_upper)
address     = raw["Address"].map(squeeze_upper)
license_num = raw["License #"].fillna("").str.strip().replace("", pd.NA)
zip_code    = raw["Zip"].fillna("").str.strip().replace("", pd.NA)
latitude    = pd.to_numeric(raw["Latitude"], errors="coerce")
longitude   = pd.to_numeric(raw["Longitude"], errors="coerce")

for name, b_s, a_s in [("DBA Name", raw["DBA Name"], dba_name),
                       ("AKA Name", raw["AKA Name"], aka_name),
                       ("Address",  raw["Address"],  address)]:
    print("%-10s cells changed by trim/upper: %s" % (name, format(changed_count(b_s, a_s), ",")))

print("\nzip not 5 digits :", int((~zip_code.dropna().str.fullmatch(r"\d{5}")).sum()))
print("zip missing      :", int(zip_code.isna().sum()))
print("coords missing   :", int(latitude.isna().sum()))
print("lat/long outside the Chicago bounding box (41.6-42.1, -87.95..-87.5):",
      int((latitude.notna() & (~latitude.between(41.6, 42.1) | ~longitude.between(-87.95, -87.5))).sum()))

log_op("2.6 Identity/address", "trim_identity_fields",
       "Whitespace squeeze + upper-case on DBA/AKA/Address; trim on License #/Zip; lat/long -> float.",
       "USEFUL only. U1 does not group by establishment or address, so per the scope guardrail we do "
       "the free normalization and explicitly do NOT attempt entity resolution or impute geo values.",
       before={"dba_untrimmed_cells": changed_count(raw["DBA Name"], dba_name)},
       after={"dba_untrimmed_cells": 0},
       rows_affected=changed_count(raw["DBA Name"], dba_name) + changed_count(raw["Address"], address),
       columns_in=["DBA Name", "AKA Name", "Address", "License #", "Zip", "Latitude", "Longitude"],
       columns_out=["dba_name", "aka_name", "address", "license_num", "zip", "latitude", "longitude"])

DBA Name   cells changed by trim/upper: 22,607
AKA Name   cells changed by trim/upper: 21,991
Address    cells changed by trim/upper: 17,887

zip not 5 digits : 0
zip missing      : 42
coords missing   : 1022
lat/long outside the Chicago bounding box (41.6-42.1, -87.95..-87.5): 0
[op009] 2.6 Identity/address :: trim_identity_fields  |  before={'dba_untrimmed_cells': 22607} -> after={'dba_untrimmed_cells': 0}; rows_affected=40,494


---
# Step 3 — `Facility Type` → `facility_group`

**What.** Collapse **520 distinct** raw facility strings (plus 5,264 blanks) into a small controlled
vocabulary of **16 groups** through a three-tier cascade:

1. an **exact map** for the high-volume canonical values (`Restaurant`, `Grocery Store`, …);
2. **ordered keyword rules** (word-boundary-aware regex) for the long tail — the order *is* a
   documented precedence;
3. anything still unmatched → `Other`; blank → `Unknown`.

`facility_type_raw` is retained beside `facility_group` so any grouping decision can be revisited
without re-cleaning.

**Why for U1.** ***Necessary — U1 breaks results down by facility type.*** With 520 raw values the
breakdown has a long tail of one-row "types" (`REST/GROCERY`, `Restuarant`,
`GROCERY STORE/ GAS STATION`, …), which makes the answer unreadable and statistically meaningless.
This grouping is what turns "facility type" into an actual analysis dimension.

**Documented precedence rule.** Composite types are common (`GROCERY STORE/RESTAURANT`,
`RESTAURANT/BAR`, `GAS STATION/GROCERY`). We resolve them **prepared-food-service first**:
shared kitchens → child care & health facilities (highest regulatory sensitivity) → schools →
mobile → bakery → catering → restaurant → retail grocery → bar → convenience. Rationale: U1 is about
*food-handling violations*, which attach to the food-preparation activity on the premises, so a
place that both sells groceries and cooks is best characterized by the cooking.

In [16]:
FACILITY_EXACT = {
    "RESTAURANT": "Restaurant", "GROCERY STORE": "Grocery Store", "SCHOOL": "School",
    "CHILDREN'S SERVICES FACILITY": "Childcare / Daycare", "BAKERY": "Bakery",
    "DAYCARE ABOVE AND UNDER 2 YEARS": "Childcare / Daycare",
    "DAYCARE (2 - 6 YEARS)": "Childcare / Daycare", "DAYCARE (UNDER 2 YEARS)": "Childcare / Daycare",
    "LONG TERM CARE": "Long Term Care / Health", "HOSPITAL": "Long Term Care / Health",
    "CATERING": "Catering / Banquet", "LIQUOR": "Bar / Tavern / Liquor", "TAVERN": "Bar / Tavern / Liquor",
    "MOBILE FOOD PREPARER": "Mobile Food", "MOBILE FOOD DISPENSER": "Mobile Food",
    "GOLDEN DINER": "Restaurant", "WHOLESALE": "Wholesale / Warehouse / Commissary",
    "SPECIAL EVENT": "Special Event / Venue", "SHELTER": "Social Service / Shelter",
}

# ORDER IS THE PRECEDENCE RULE - first match wins. Rationale in the markdown cell above.
FACILITY_RULES = [
    ("Wholesale / Warehouse / Commissary",
     r"SHARED KITCHEN|COMMISSARY|COMMISARY|COMMIASARY"),
    ("Childcare / Daycare",
     r"DAY ?CARE|CHILDER|CHILDREN|CHILD CARE|\b1023\b|AFTER SCHOOL|PRESCHOOL|PRE-SCHOOL|HEAD START|NURSERY|YOUTH|MONTS|MONTHS TO|YRS TO|YEARS OLD"),
    ("Long Term Care / Health",
     r"LONG ?-?TERM|NURSING|ASSIS+TED LIVING|HOSPITAL|REHAB|SENIOR|SUPPORTIVE LIVING|HEALTH ?CARE|DIALYSIS|MEDICAL|CARE FACILITY|CARE CENTER"),
    ("School",
     r"SCHOOL|SHCOOL|COLLEGE|UNIVERSIT|ACADEM|CHARTER|CULINARY|COOKING CLASS|TEACHING|EDUCATION|CAMPUS|INSTITUTE|PASTRY SCH|CHEF TRAINING"),
    ("Mobile Food",
     r"MOBILE|MOBIL |MOBILP|\bMFD\b|FOOD TRUCK|PUSH ?CART|\bCART\b|PEDDLER|VENDING|ICE CREAM TRUCK"),
    ("Bakery",
     r"BAKERY|BAKING|PASTRY|DONUT|DOUGHNUT|BAGEL"),
    ("Catering / Banquet",
     r"CATER|BANQUET|EVENT SPACE|EVENT CENTER"),
    ("Restaurant",
     r"RESTAURANT|RESTUARANT|RESTURANT|RSTAURANT|\bREST\b|CAFE\b|COFFEE|DINER|DINING|JUICE|SMOOTHIE|PIZZA|TAQUERIA|SUSHI|HOT DOG|SNACK|\bDELI\b|ICE CREAM|GELATO|PALETERIA|CANDY|CHOCOLATE|FROZEN DESSERT|BUBBLE|FOOD COURT|EATERY|GRILL|BUFFET|SANDWICH|BBQ|SALAD|POPCORN|MILK TEA|SHAKE"),
    ("Grocery Store",
     r"GROCERY|SUPERMARKET|\bMARKET\b|BUTCHER|MEAT|PRODUCE|FISH|POULTRY|SLAUGHTER|BODEGA|MINI ?MART|FRUIT|VEGETAB"),
    ("Bar / Tavern / Liquor",
     r"LIQUOR|TAVERN|\bBARS?\b|BREWERY|BREW ?PUB|\bPUB\b|WINE|NIGHT ?CLUB|LOUNGE|COCKTAIL|DISTILLER"),
    ("Convenience / Gas Station",
     r"GAS STATION|CONVENIEN|\bKIOSK\b|DOLLAR|DRUG ?STORE|PHARMAC|\bSTORE\b|RETAIL|NEWS ?STAND|TOBACCO|GIFT SHOP"),
    ("Wholesale / Warehouse / Commissary",
     r"WHOLESALE|DISTRIBUT|WAREHOUSE|PACKAG|PROCESS|MANUFACTUR|SUPPLIER|IMPORT|COLD STORAGE|\bSTORAGE\b|KITCHEN"),
    ("Institutional Dining",
     r"CAFETERIA|DINING HALL|EMPLOYEE|CANTEEN|MESS HALL|FEEDING"),
    ("Social Service / Shelter",
     r"SHELTER|CHURCH|PANTRY|CHARIT|SOUP|NOT-?FOR-?PROFIT|NON ?-?PROFIT|MISSION|HOUSING|TEMPLE|SYNAGOG|MOSQUE|RELIGIOUS|OUTREACH|ARCHDIOCESE"),
    ("Special Event / Venue",
     r"SPECIAL EVENT|STADIUM|THEAT|ROOF ?TOPS?|POP-?UP|FESTIVAL|AIRPORT|HOTEL|HOSTEL|FITNESS|\bGYM\b|VENUE|VENU\b|\bCLUB\b|GOLF|BOWL|CASINO|ARENA|\bPARK\b|\bZOO\b|MUSEUM|MUSIC|LIVE PERFORMANCE|\bHALL\b|CRUISE|\bBOAT\b|CONCESSION|NAVY PIER|RIVERWALK|NORTHERLY ISLAND|WRIGLEY"),
]

def facility_group(s):
    # returns (facility_group, rule_id)
    if s is None or isinstance(s, float) or str(s).strip() == "":
        return "Unknown", "F0_blank_to_Unknown"
    u = re.sub(r"\s+", " ", str(s).upper()).strip()
    if u in FACILITY_EXACT:
        return FACILITY_EXACT[u], "F1_exact_map"
    for grp, pat in FACILITY_RULES:
        if re.search(pat, u):
            return grp, "F2_keyword:" + grp
    return "Other", "F3_unmatched"

# evaluate once per DISTINCT raw value (521 regex passes instead of 298,869)
_vals         = raw["Facility Type"].fillna("\x00")
_lookup       = {v: facility_group(None if v == "\x00" else v) for v in _vals.unique()}
facility_grp  = _vals.map(lambda v: _lookup[v][0])
facility_rule = _vals.map(lambda v: _lookup[v][1])

summary = (pd.DataFrame({"facility_group": facility_grp, "raw": raw["Facility Type"]})
           .groupby("facility_group")
           .agg(n_rows=("raw", "size"), n_distinct_raw_values=("raw", "nunique"))
           .sort_values("n_rows", ascending=False))
summary["pct_of_D"] = (summary.n_rows / len(raw) * 100).round(2)
print("distinct facility strings: %d (+ blank)  ->  %d groups"
      % (raw["Facility Type"].nunique(), facility_grp.nunique()))
print(summary.to_string())

distinct facility strings: 520 (+ blank)  ->  16 groups
                                    n_rows  n_distinct_raw_values  pct_of_D
facility_group                                                             
Restaurant                          203457                     81     68.08
Grocery Store                        36772                     49     12.30
School                               19418                     32      6.50
Childcare / Daycare                  15577                     36      5.21
Unknown                               5264                      0      1.76
Bakery                                4376                     12      1.46
Long Term Care / Health               3447                     16      1.15
Mobile Food                           2491                     31      0.83
Catering / Banquet                    2417                     25      0.81
Bar / Tavern / Liquor                 1888                     37      0.63
Wholesale / Warehouse / Commissa

In [17]:
print("Rows that no rule could classify (facility_group='Other'): %d  (%.3f%% of D)"
      % (int((facility_grp == "Other").sum()), (facility_grp == "Other").mean() * 100))
print("\nthe unmapped raw values (top 30):")
print(raw.loc[facility_grp == "Other", "Facility Type"].value_counts().head(30).to_string())

print("\n\nSpot-check of the precedence rule on composite / misspelled values:")
spot = ["GROCERY STORE/GAS STATION", "RESTAURANT/BAR", "GROCERY/RESTAURANT", "Restuarant",
        "Shared Kitchen User (Long Term)", "1023 CHILDERN'S SERVICES FACILITY", "PUBLIC SHCOOL",
        "CHARTER SCHOOL CAFETERIA", "MOBILE FOOD TRUCK", "WRIGLEY ROOFTOP", "LIVE POULTRY",
        "Pop-Up Establishment Host-Tier II", "grocery/butcher", "COFFEE CART"]
print(pd.DataFrame([(s,) + facility_group(s) for s in spot],
                   columns=["raw value", "facility_group", "rule"]).to_string(index=False))

log_op("3. Facility type", "map_facility_type_to_group",
       "Collapsed 520 raw Facility Type strings into a 16-value controlled vocabulary using an exact "
       "map plus an ordered keyword cascade; blank -> 'Unknown', unmatched -> 'Other'.",
       "NECESSARY: U1 breaks the violation counts down BY FACILITY TYPE. A 520-value dimension with a "
       "one-row long tail makes that breakdown unreadable and statistically meaningless.",
       params={"n_exact_map_entries": len(FACILITY_EXACT), "n_keyword_rules": len(FACILITY_RULES),
               "precedence": "food-preparation activity beats retail activity for composite types"},
       before={"distinct_values": int(raw["Facility Type"].nunique()),
               "blank_rows": int(raw["Facility Type"].isna().sum())},
       after={"distinct_values": int(facility_grp.nunique()),
              "unmapped_Other_rows": int((facility_grp == "Other").sum()),
              "Unknown_rows": int((facility_grp == "Unknown").sum())},
       rows_affected=changed_count(raw["Facility Type"], facility_grp),
       columns_in=["Facility Type"], columns_out=["facility_type_raw", "facility_group"])

Rows that no rule could classify (facility_group='Other'): 190  (0.064% of D)

the unmapped raw values (top 30):
Facility Type
HERBALIFE              27
Other                  18
REGULATED BUSINESS     12
TRUCK                   7
HERBAL LIFE SHOP        7
ROOM SERVICE            7
HERBALIFE/ZUMBA         6
HERBAL MEDICINE         6
HERBAL LIFE             6
Pool                    6
HERBAL DRINKS           5
SMOKEHOUSE              5
HERBAL REMEDY           4
Herabalife              4
Illegal Vendor          3
HEALTH CENTER           3
HERBALCAL               3
watermelon house        3
weight loss program     3
Herbalife Nutrition     3
Laundromat              3
CHINESE HERBS           3
FOOD BOOTH              3
blockbuster video       3
ADULT DAY SERVICE       3
GREENHOUSE              2
HELICOPTER TERMINAL     2
NUTRITION/HERBALIFE     2
URBAN FARM              2
NOT FOR PROFIT          2


Spot-check of the precedence rule on composite / misspelled values:
                       

---
# Step 4 — Parse `Violations` into a violation-level relation ⭐ *the critical step*

**What.** The `Violations` column packs **every** violation found during an inspection into a single
free-text cell, `|`-separated, each chunk shaped
`<NUMBER>. <TITLE> - Comments: <inspector comment>`. We split on `|`, regex-parse each chunk, and
emit **one row per violation per inspection** into a second table `clean_violations`, keyed by
`inspection_id` and `violation_seq` (which preserves the original order inside the cell).

**Why for U1.** ***This is the step U1 rests on.*** U1 asks for *the most common violation
categories*. In `D` a "violation category" is not a value anywhere — it is a substring of a free-text
blob shared with up to 40 other violations and their inspector narratives. No `GROUP BY` can reach
it. Splitting `D` into `clean_inspections` ⋈ `clean_violations` puts the data into **first normal
form** and turns U1 from a text-mining problem into a two-line SQL query.

**Design decisions, and why:**

| Decision | Choice | Reason |
|---|---|---|
| Blank `Violations` | contributes **zero** violation rows | An absent violation list is not "a violation with an empty name". Recorded on the inspection side as `has_violation_text = False`. |
| Repeated violation number within one inspection (59,359 redundant rows across 33,317 inspections) | **keep every parsed row** | The repeats carry *different inspector comments*, i.e. genuinely distinct observations. Deleting them would destroy provenance. U1's "how many inspections cited category X" is answered with `COUNT(DISTINCT inspection_id)` in the query instead — see Step 9. |
| Out-of-domain violation numbers | flag, never drop | Guardrail: never delete to fix. |
| Category key | the **title**, qualified by `code_era` | See the code-book finding below. |

**Finding that changed the plan — the code book changed on 2018-07-01.** The Phase-I plan assumed
violation numbers run 1–44 plus 70. They do not: the data contains **1–64 plus 70**, because the
City of Chicago replaced its food-code checklist in July 2018. The same *number* denotes different
*violations* on either side of that date, e.g.

* `3.` before 2018-07-01 = *"POTENTIALLY HAZARDOUS FOOD MEETS TEMPERATURE REQUIREMENT…"*
* `3.` after 2018-07-01 = *"MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE…"*

Grouping U1 by violation *number* alone would therefore silently merge unrelated categories across
the 2018 boundary. We verify this below (it is a functional-dependency violation), carry `code_era`
on every violation row, and validate numbers against the **era-specific** domain
(`pre_2018_code`: 1–45 ∪ {70}; `2018_code`: 1–64).

In [18]:
VIOLATION_RE = re.compile(r"^\s*(\d+)\.\s*(.*?)\s*(?:-\s*Comments:\s*(.*))?$", re.S)

def parse_violation_cell(cell):
    # one free-text Violations cell -> list of (seq, number, title, comment, parse_ok)
    out = []
    for seq, chunk in enumerate(str(cell).split("|"), start=1):
        chunk = chunk.strip()
        if chunk == "":
            continue
        m = VIOLATION_RE.match(chunk)
        if m:
            out.append((seq, int(m.group(1)),
                        re.sub(r"\s+", " ", m.group(2)).strip(),
                        re.sub(r"\s+", " ", m.group(3) or "").strip(), True))
        else:
            out.append((seq, pd.NA, re.sub(r"\s+", " ", chunk)[:250], "", False))
    return out

t0, rows, n_cells, n_unparsed = time.time(), [], 0, 0
for iid, era, cell in zip(raw["Inspection ID"], code_era, raw["Violations"]):
    if cell is None or isinstance(cell, float):
        continue
    n_cells += 1
    for seq, num, ttl, cmt, ok in parse_violation_cell(cell):
        rows.append((iid, seq, num, ttl, cmt, era, ok))
        if not ok:
            n_unparsed += 1

violations = pd.DataFrame(rows, columns=["inspection_id", "violation_seq", "violation_number",
                                         "violation_title", "violation_comment",
                                         "code_era", "parse_ok"])
violations["violation_number"] = violations["violation_number"].astype("Int64")
del rows

print("parsed %s non-empty Violations cells in %.1fs" % (format(n_cells, ","), time.time() - t0))
print("-> %s atomic violation rows" % format(len(violations), ","))
print("unparsed chunks: %d  (parse success rate: %.4f%%)"
      % (n_unparsed, (1 - n_unparsed / max(len(violations), 1)) * 100))
violations.head(6)

parsed 215,525 non-empty Violations cells in 9.4s
-> 974,597 atomic violation rows
unparsed chunks: 0  (parse success rate: 100.0000%)


,inspection_id,violation_seq,violation_number,violation_title,violation_comment,code_era,parse_ok
0,2625743,1,53,"TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED",OBSERVED UNISEX TOILET ROOM DOOR NOT SELF CLOSING. INSTRUCTED TO INSTALL A SELF CLOSIN...,2018_code,True
1,2625743,2,58,ALLERGEN TRAINING AS REQUIRED,OBSERVED NO ALLERGEN TRAINING ON SITE DURING TIME OF INSPECTION. INSTRUCTED TO OBTAIN ...,2018_code,True
2,2625757,1,10,ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE,HANDSINK IN REAR PREP AREA IN POOR REPAIR AT THIS TIME. SINK IS VERY SLOW DRAINING. IN...,2018_code,True
3,2625757,2,40,PERSONAL CLEANLINESS,FOOD HANDLER IN NEED OF EFFECTIVE HAIR RESTRAINT. INSTRUCTED TO CORRECT AND MAINTAIN,2018_code,True
4,2625757,3,47,"FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED",WORN TORN GASKETS IN NEED OF REPLACING ON FRONT REACH IN COOLER. REAR REACH IN COOLER ...,2018_code,True
5,2625757,4,51,PLUMBING INSTALLED; PROPER BACKFLOW DEVICES,OBSERVED LEAK AT FAUCET ON 3 COMPARTMENT SINK. OBSERVED LEAK AT PROOFER OVEN INSTRUCTE...,2018_code,True


In [19]:
# --------------------------------------------- evidence for the 2018 code-book change
fd_number_only = violations.groupby("violation_number")["violation_title"].nunique()
fd_era_number  = violations.groupby(["code_era", "violation_number"])["violation_title"].nunique()

print("Functional dependency   violation_number -> violation_title")
print("  numbers mapping to >1 distinct title  :", int((fd_number_only > 1).sum()), " <-- FD VIOLATED")
print("\nFunctional dependency   (code_era, violation_number) -> violation_title")
print("  (era, number) pairs mapping to >1 title:", int((fd_era_number > 1).sum()), " <-- FD HOLDS")
print("\ndistinct (era, number) pairs:", len(fd_era_number),
      "| distinct titles:", violations["violation_title"].nunique())
print("-> the mapping is a bijection, so the TITLE is itself a safe category key across both eras.")

print("\nWorked example - violation number 3 denotes two different things:")
print(violations[violations.violation_number == 3]
      .groupby(["code_era", "violation_title"]).size().rename("n_rows").to_string())

def compact_ranges(nums):
    # [1,2,3,45,70] -> '1-3, 45, 70'
    out, start, prev = [], nums[0], nums[0]
    for n in nums[1:]:
        if n == prev + 1:
            prev = n
            continue
        out.append((start, prev)); start = prev = n
    out.append((start, prev))
    return ", ".join(str(a) if a == b else "%d-%d" % (a, b) for a, b in out)

print("\nViolation numbers actually observed per era (this is the era-specific code book):")
for era, g in violations.groupby("code_era"):
    nums = sorted(int(n) for n in g["violation_number"].dropna().unique())
    print("  %-15s: %-18s (%d distinct codes)" % (era, compact_ranges(nums), len(nums)))

Functional dependency   violation_number -> violation_title
  numbers mapping to >1 distinct title  : 45  <-- FD VIOLATED

Functional dependency   (code_era, violation_number) -> violation_title
  (era, number) pairs mapping to >1 title: 0  <-- FD HOLDS

distinct (era, number) pairs: 110 | distinct titles: 110
-> the mapping is a bijection, so the TITLE is itself a safe category key across both eras.

Worked example - violation number 3 denotes two different things:
code_era       violation_title                                                                                         
2018_code      MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITIES AND REPORTING               18124
pre_2018_code  POTENTIALLY HAZARDOUS FOOD MEETS TEMPERATURE REQUIREMENT DURING STORAGE, PREPARATION DISPLAY AND SERVICE     9028

Violation numbers actually observed per era (this is the era-specific code book):
  2018_code      : 1-64               (64 distinct codes)
  pre_2018

In [20]:
# --------------------------------------------- validate against the era-specific domain
VALID_CODES = {
    "pre_2018_code": set(range(1, 46)) | {70},   # 1-45 plus '70. NO SMOKING REGULATIONS'
    "2018_code":     set(range(1, 65)),          # 1-64
}

def code_valid(era, num):
    if pd.isna(num):
        return False
    return int(num) in VALID_CODES.get(era, set())

violations["violation_number_valid"] = [code_valid(e, n) for e, n
                                        in zip(violations.code_era, violations.violation_number)]
violations["violation_key"] = (violations.code_era.str.replace("_code", "", regex=False) + "-"
                               + violations.violation_number.astype("string").fillna("NA"))

n_bad = int((~violations.violation_number_valid).sum())
print("violation rows outside their era's code domain:", n_bad)
if n_bad:
    print(violations[~violations.violation_number_valid]
          .groupby(["code_era", "violation_number"], dropna=False).size().to_string())

print("\nrows with an empty inspector comment: %s (%.2f%%)  <- allowed: not every citation carries "
      "narrative text" % (format(int((violations.violation_comment == "").sum()), ","),
                          (violations.violation_comment == "").mean() * 100))

print("\nviolations per inspection (over inspections that have any):")
print(violations.groupby("inspection_id").size().describe().to_string())

dup_extra = int(violations.duplicated(["inspection_id", "violation_number"]).sum())
dup_all   = violations.duplicated(["inspection_id", "violation_number"], keep=False)
print("\nrepeated (inspection_id, violation_number): %s redundant rows beyond the first citation "
      "(%s rows involved in total) across %s inspections"
      % (format(dup_extra, ","), format(int(dup_all.sum()), ","),
         format(int(violations.loc[dup_all, "inspection_id"].nunique()), ",")))
print("-> kept in clean_violations (they carry different comments); de-duplicated AT QUERY TIME with")
print("   COUNT(DISTINCT inspection_id), so U1 counts each category once per inspection.")

log_op("4. Violations", "parse_multivalued_violations_to_relation",
       "Split the multi-valued Violations cell on '|' and regex-parsed each chunk into "
       "(violation_number, violation_title, violation_comment), emitting one row per violation per "
       "inspection into a second relation; attached code_era and validated numbers per era.",
       "THE core step for U1: 'violation category' does not exist as a value in D - it is a substring "
       "of a free-text blob. Normalizing to a violation-level relation makes U1 a GROUP BY.",
       params={"separator": "|", "regex": VIOLATION_RE.pattern,
               "valid_codes": {"pre_2018_code": "1-45 and 70", "2018_code": "1-64"},
               "duplicate_policy": "keep all rows; de-duplicate at query time"},
       before={"atomic_violation_rows": 0, "cells_with_packed_violations": n_cells,
               "queryable_category_field": False},
       after={"atomic_violation_rows": len(violations), "unparsed_chunks": n_unparsed,
              "distinct_categories": int(violations.violation_title.nunique()),
              "queryable_category_field": True},
       rows_affected=len(violations),
       columns_in=["Violations", "Inspection ID", "Inspection Date"],
       columns_out=["inspection_id", "violation_seq", "violation_number", "violation_title",
                    "violation_comment", "code_era", "violation_number_valid"])

violation rows outside their era's code domain: 0

rows with an empty inspector comment: 1,730 (0.18%)  <- allowed: not every citation carries narrative text

violations per inspection (over inspections that have any):
count    215525.000000
mean          4.521967
std           3.194940
min           1.000000
25%           2.000000
50%           4.000000
75%           6.000000
max          40.000000



repeated (inspection_id, violation_number): 59,359 redundant rows beyond the first citation (106,061 rows involved in total) across 33,317 inspections
-> kept in clean_violations (they carry different comments); de-duplicated AT QUERY TIME with
   COUNT(DISTINCT inspection_id), so U1 counts each category once per inspection.
[op011] 4. Violations :: parse_multivalued_violations_to_relation  |  before={'atomic_violation_rows': 0, 'cells_with_packed_violations': 215525, 'queryable_category_field': False} -> after={'atomic_violation_rows': 974597, 'unparsed_chunks': 0, 'distinct_categories': 110, 'queryable_category_field': True}; rows_affected=974,597


In [21]:
# --------------------------------------------- the recovered category vocabulary
cat = (violations.groupby(["code_era", "violation_number", "violation_title"])
       .size().rename("n").reset_index().sort_values(["code_era", "violation_number"]))
print("Violation category vocabulary recovered from free text: %d categories\n" % len(cat))
print(cat.head(30).to_string(index=False))
print("   ... (%d more)" % (len(cat) - 30))

Violation category vocabulary recovered from free text: 110 categories

 code_era  violation_number                                                                               violation_title     n
2018_code                 1                         PERSON IN CHARGE PRESENT, DEMONSTRATES KNOWLEDGE, AND PERFORMS DUTIES  4552
2018_code                 2                                           CITY OF CHICAGO FOOD SERVICE SANITATION CERTIFICATE  9622
2018_code                 3 MANAGEMENT, FOOD EMPLOYEE AND CONDITIONAL EMPLOYEE; KNOWLEDGE, RESPONSIBILITIES AND REPORTING 18124
2018_code                 4                                                       PROPER USE OF RESTRICTION AND EXCLUSION    15
2018_code                 5                                    PROCEDURES FOR RESPONDING TO VOMITING AND DIARRHEAL EVENTS 19005
2018_code                 6                                              PROPER EATING, TASTING, DRINKING, OR TOBACCO USE   552
2018_code                 7     

---
# Step 5 — Missing values and quality flags

**What.** Classify every remaining blank as *acceptable* or *problematic* and attach an explicit
boolean flag instead of imputing or deleting:

| Flag | Meaning | Treatment for U1 |
|---|---|---|
| `has_violation_text` | the inspection carried any violation text | — |
| `n_violations` | number of parsed violation rows | — |
| `fail_missing_violations` | `result = 'Fail'` **and** no violation record | **Excluded** from U1's denominator, reported separately |
| `risk_is_imputed` | risk was blank or the undocumented `All` | reported; sits in the `Unknown` bucket |
| `facility_type_missing` | facility type blank → `Unknown` group | kept, shown as `Unknown` in the breakdown |
| `missing_city`, `missing_state`, `out_of_state` | scope / location gaps | reported, not deleted |
| `missing_zip`, `missing_geo` | location gaps | kept; `missing_geo` rows are simply excluded from maps |

**Why for U1.** *Necessary.* A failed inspection with no violation text is not "a failure with zero
violations" — it is **missing data masquerading as a zero**, and if it silently enters U1 it deflates
every category's rate. Making it a flag lets the analysis query exclude it *and* lets the report
state exactly how many such records exist. Every other flag exists so U1's result can be qualified
rather than quietly biased.

In [22]:
viol_counts             = violations.groupby("inspection_id").size()
n_violations            = raw["Inspection ID"].map(viol_counts).fillna(0).astype(int)
has_violation_text      = raw["Violations"].notna()
fail_missing_violations = (result_clean == "Fail") & (n_violations == 0)

flags = pd.DataFrame({
    "has_violation_text":      has_violation_text,
    "n_violations":            n_violations,
    "fail_missing_violations": fail_missing_violations,
    "risk_is_imputed":         risk_is_imputed,
    "facility_type_missing":   raw["Facility Type"].isna(),
    "missing_city":            raw["City"].isna() | (city_clean == "UNKNOWN"),
    "missing_state":           state_clean == "UNKNOWN",
    "out_of_state":            out_of_state,
    "missing_zip":             zip_code.isna(),
    "missing_geo":             latitude.isna() | longitude.isna(),
})

rep = pd.DataFrame({"n_rows_flagged": flags.drop(columns=["n_violations"]).sum()})
rep["pct_of_D"] = (rep.n_rows_flagged / len(raw) * 100).round(3)
rep["classification"] = ["ok (absence is meaningful)",
                         "PROBLEM (missing data as a false zero)",
                         "assumption (blank or 'All' risk)",
                         "ok (bucketed as Unknown)",
                         "ok (scope note)", "ok (scope note)", "ok (scope note)",
                         "ok (not fabricated)", "ok (excluded from maps only)"]
print(rep.to_string())

print("\nfail_missing_violations - are these really failures with no findings?")
_fmv = raw.loc[fail_missing_violations.values]
print("  count: %s | share of all Fail rows: %.2f%%"
      % (format(len(_fmv), ","), len(_fmv) / max(int((result_clean == "Fail").sum()), 1) * 100))
print("  by year:")
print(_fmv["Inspection Date"].str[-4:].value_counts().sort_index().to_string())

log_op("5. Missing values", "classify_missing_and_flag",
       "Derived 10 explicit quality flags instead of imputing or deleting: has_violation_text, "
       "n_violations, fail_missing_violations, risk_is_imputed, facility_type_missing, "
       "missing_city/state/zip/geo, out_of_state.",
       "NECESSARY: a Fail with no violation text is missing data, not zero violations. Flagging lets "
       "U1 exclude those rows explicitly instead of silently deflating every category rate.",
       before={"explicit_quality_flags": 0},
       after={"explicit_quality_flags": int(flags.shape[1]),
              "fail_missing_violations": int(fail_missing_violations.sum())},
       rows_affected=int(flags.drop(columns=["n_violations", "has_violation_text"]).any(axis=1).sum()),
       columns_in=["Violations", "Results", "Risk", "Facility Type", "City", "State", "Zip",
                   "Latitude", "Longitude"],
       columns_out=list(flags.columns))

                         n_rows_flagged  pct_of_D                          classification
has_violation_text               215525    72.114              ok (absence is meaningful)
fail_missing_violations            3579     1.198  PROBLEM (missing data as a false zero)
risk_is_imputed                     167     0.056        assumption (blank or 'All' risk)
facility_type_missing              5264     1.761                ok (bucketed as Unknown)
missing_city                        174     0.058                         ok (scope note)
missing_state                        58     0.019                         ok (scope note)
out_of_state                         18     0.006                         ok (scope note)
missing_zip                          42     0.014                     ok (not fabricated)
missing_geo                        1022     0.342            ok (excluded from maps only)

fail_missing_violations - are these really failures with no findings?
  count: 3,579 | share of all

---
# Step 6 — Assemble and emit `D'`

**What.** Build `clean_inspections` (exactly one row per `inspection_id`: cleaned columns, retained
`*_raw` originals, and the flags) and `clean_violations` (the 1NF fan-out), assert the schema
invariants, and write both to CSV.

**Why for U1.** *Necessary* — this **is** `D'`. The invariants asserted here (`|clean_inspections| =
|D|`, `inspection_id` unique, no orphan violations) are the guarantees the rest of the report leans
on: no row of `D` was destroyed, and the only structural change is the intended 1→N decomposition of
the violation column.

In [23]:
clean_inspections = pd.DataFrame({
    # ---- key
    "inspection_id":       raw["Inspection ID"],
    # ---- establishment
    "dba_name":            dba_name,
    "aka_name":            aka_name,
    "license_num":         license_num,
    "facility_type_raw":   raw["Facility Type"],
    "facility_group":      facility_grp,
    # ---- risk
    "risk_raw":            raw["Risk"],
    "risk_clean":          risk_clean,
    "risk_level":          risk_level,
    # ---- location
    "address":             address,
    "city_raw":            raw["City"],
    "city_clean":          city_clean,
    "state_raw":           raw["State"],
    "state_clean":         state_clean,
    "zip":                 zip_code,
    "latitude":            latitude,
    "longitude":           longitude,
    # ---- inspection
    "inspection_date":     inspection_date.dt.strftime("%Y-%m-%d"),
    "inspection_year":     inspection_year,
    "code_era":            code_era,
    "inspection_type_raw": raw["Inspection Type"],
    "inspection_type":     inspection_type,
    "is_reinspection":     is_reinspection,
    "result":              result_clean,
})
clean_inspections = pd.concat([clean_inspections, flags], axis=1)

clean_violations = violations[["inspection_id", "violation_seq", "violation_number",
                               "violation_title", "violation_comment", "code_era",
                               "violation_key", "violation_number_valid"]].copy()

# ---------------------------------------------------------------- invariants
assert len(clean_inspections) == len(raw), "row count changed!"
assert clean_inspections["inspection_id"].is_unique, "inspection_id is not a key!"
assert set(clean_violations["inspection_id"]) <= set(clean_inspections["inspection_id"]), "orphans!"
print("INVARIANTS OK")
print("  clean_inspections : %s rows x %d cols   (D had %s rows x %d cols)"
      % (format(len(clean_inspections), ","), clean_inspections.shape[1],
         format(len(raw), ","), raw.shape[1]))
print("  clean_violations  : %s rows x %d cols"
      % (format(len(clean_violations), ","), clean_violations.shape[1]))
clean_inspections.head(3)

INVARIANTS OK
  clean_inspections : 298,869 rows x 34 cols   (D had 298,869 rows x 17 cols)
  clean_violations  : 974,597 rows x 8 cols


,inspection_id,dba_name,aka_name,license_num,facility_type_raw,facility_group,risk_raw,risk_clean,risk_level,address,...,has_violation_text,n_violations,fail_missing_violations,risk_is_imputed,facility_type_missing,missing_city,missing_state,out_of_state,missing_zip,missing_geo
0,2625743,XOCOBERRY,XOCOBERRY,3046764,Restaurant,Restaurant,Risk 1 (High),Risk 1 (High),1,5158 S KEDZIE AVE,...,True,2,False,False,False,False,False,False,False,False
1,2625757,"MARZEYA BAKERY J.A.S., INC.","MARZEYA BAKERY J.A.S., INC.",2712762,Restaurant,Restaurant,Risk 2 (Medium),Risk 2 (Medium),2,8908 S COMMERCIAL AVE,...,True,4,False,False,False,False,False,False,False,False
2,2625731,PATHWAYS TO LEARNING CHILD CARE INC.,PATHWAYS TO LEARNING CHILD CARE INC.,2215782,Daycare Above and Under 2 Years,Childcare / Daycare,Risk 1 (High),Risk 1 (High),1,6535 S KEDZIE AVE,...,True,3,False,False,False,False,False,False,False,False


In [24]:
p_insp = os.path.join(OUT_DIR, "clean_inspections.csv")
p_viol = os.path.join(OUT_DIR, "clean_violations.csv")
t0 = time.time()
clean_inspections.to_csv(p_insp, index=False)
clean_violations.to_csv(p_viol, index=False)
print("written in %.1fs" % (time.time() - t0))
for p in (p_insp, p_viol):
    print("  %-26s %8.1f MB" % (os.path.basename(p), os.path.getsize(p) / 1e6))

log_op("6. Emit", "write_cleaned_datasets",
       "Assembled and wrote clean_inspections.csv (1 row per inspection) and clean_violations.csv "
       "(1 row per violation per inspection).",
       "These two relations ARE D'. The 1:N split is the structural change that makes U1 expressible.",
       before={"relations": 1, "rows": len(raw), "columns": raw.shape[1]},
       after={"relations": 2, "inspection_rows": len(clean_inspections),
              "inspection_columns": clean_inspections.shape[1],
              "violation_rows": len(clean_violations)},
       rows_affected=len(clean_inspections) + len(clean_violations),
       columns_out=list(clean_inspections.columns) + list(clean_violations.columns))

written in 5.2s
  clean_inspections.csv          91.7 MB
  clean_violations.csv          285.0 MB
[op013] 6. Emit :: write_cleaned_datasets  |  before={'relations': 1, 'rows': 298869, 'columns': 17} -> after={'relations': 2, 'inspection_rows': 298869, 'inspection_columns': 34, 'violation_rows': 974597}; rows_affected=1,273,466


---
# Step 7 — Integrity-constraint checks in SQL (DuckDB): `Q_U(D)` vs `Q_U(D')`

**What.** Fourteen integrity constraints, each expressed as a SQL query returning the **number of
violating rows**, run over the **raw** table and over the **cleaned** tables. The identical SQL text
is exported to `queries.txt`.

**Why for U1.** *Necessary as evidence.* "The data is cleaner" is an assertion; a denial-constraint
violation count that drops from 167 to 0 is a measurement. The constraints were chosen to cover
exactly the properties U1 depends on: a valid key, closed domains for the two grouping dimensions
(`risk`, `facility_group`), first normal form for the violation data, referential integrity between
the two output relations, and the functional dependency that stops violation categories merging
across the 2018 code change.

Some constraints are **not machine-checkable on `D`** — e.g. "every violation row references an
existing inspection" presupposes a violation relation, which `D` does not have. Those report
`n/a` rather than a fake zero; that they *become* checkable at all is itself part of the improvement.

In [25]:
con = duckdb.connect()
raw_sql = raw.rename(columns={
    "Inspection ID": "inspection_id", "DBA Name": "dba_name", "AKA Name": "aka_name",
    "License #": "license_num", "Facility Type": "facility_type", "Risk": "risk",
    "Address": "address", "City": "city", "State": "state", "Zip": "zip",
    "Inspection Date": "inspection_date", "Inspection Type": "inspection_type",
    "Results": "results", "Violations": "violations", "Latitude": "latitude",
    "Longitude": "longitude", "Location": "location"})
con.register("raw", raw_sql)
con.register("insp", clean_inspections)
con.register("viol", clean_violations)
print(con.sql("SELECT 'raw (D)' AS relation, COUNT(*) AS n FROM raw UNION ALL "
              "SELECT 'insp (D-prime)', COUNT(*) FROM insp UNION ALL "
              "SELECT 'viol (D-prime)', COUNT(*) FROM viol").df().to_string(index=False))

      relation      n
       raw (D) 298869
insp (D-prime) 298869
viol (D-prime) 974597


In [26]:
NA = None   # constraint not expressible on D

IC = [
dict(id="IC1", name="Key: inspection_id is unique",
     kind="key constraint", u1="required (entity identity)",
     before="SELECT COUNT(*) FROM (SELECT inspection_id FROM raw GROUP BY 1 HAVING COUNT(*)>1)",
     after ="SELECT COUNT(*) FROM (SELECT inspection_id FROM insp GROUP BY 1 HAVING COUNT(*)>1)"),

dict(id="IC2", name="Domain: Risk in {Risk 1 (High), Risk 2 (Medium), Risk 3 (Low), Unknown}",
     kind="domain constraint", u1="REQUIRED (U1 groups by risk)",
     before="SELECT COUNT(*) FROM raw\n"
            "WHERE risk IS NULL\n"
            "   OR risk NOT IN ('Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)')",
     after ="SELECT COUNT(*) FROM insp\n"
            "WHERE risk_clean IS NULL\n"
            "   OR risk_clean NOT IN ('Risk 1 (High)','Risk 2 (Medium)','Risk 3 (Low)','Unknown')"),

dict(id="IC3", name="Format: City is non-null and in canonical upper/trimmed form",
     kind="format / completeness", u1="useful (scope verification)",
     before="SELECT COUNT(*) FROM raw  WHERE city IS NULL OR city <> UPPER(TRIM(city))",
     after ="SELECT COUNT(*) FROM insp WHERE city_clean IS NULL OR city_clean <> UPPER(TRIM(city_clean))"),

dict(id="IC4", name="Consistency: no misspelled single-token variant of CHICAGO survives",
     kind="denial constraint", u1="useful (scope verification)",
     # single-token only: multi-word names such as 'CHICAGO HEIGHTS' are genuine, distinct suburbs
     before="SELECT COUNT(*) FROM raw\n"
            "WHERE UPPER(REGEXP_REPLACE(COALESCE(city,''),'[^A-Za-z]','','g')) <> 'CHICAGO'\n"
            "  AND city NOT LIKE '% %'\n"
            "  AND (UPPER(city) LIKE '%CHIC%' OR UPPER(city) LIKE '%CAGO%')",
     after ="SELECT COUNT(*) FROM insp\n"
            "WHERE UPPER(REGEXP_REPLACE(COALESCE(city_clean,''),'[^A-Za-z]','','g')) <> 'CHICAGO'\n"
            "  AND city_clean NOT LIKE '% %'\n"
            "  AND (UPPER(city_clean) LIKE '%CHIC%' OR UPPER(city_clean) LIKE '%CAGO%')"),

dict(id="IC5", name="Domain: State is a 2-letter code or explicitly UNKNOWN",
     kind="domain constraint", u1="useful (scope verification)",
     before="SELECT COUNT(*) FROM raw WHERE state IS NULL OR NOT REGEXP_MATCHES(state,'^[A-Z]{2}$')",
     after ="SELECT COUNT(*) FROM insp\n"
            "WHERE state_clean IS NULL\n"
            "   OR NOT (REGEXP_MATCHES(state_clean,'^[A-Z]{2}$') OR state_clean = 'UNKNOWN')"),

dict(id="IC6", name="Domain: Results in the 7 documented outcomes  (the U0 control)",
     kind="domain constraint", u1="required (U1 filters Results='Fail')",
     before="SELECT COUNT(*) FROM raw WHERE results IS NULL OR results NOT IN\n"
            "  ('Pass','Fail','Pass w/ Conditions','Out of Business','No Entry','Not Ready','Business Not Located')",
     after ="SELECT COUNT(*) FROM insp WHERE result IS NULL OR result NOT IN\n"
            "  ('Pass','Fail','Pass w/ Conditions','Out of Business','No Entry','Not Ready','Business Not Located')"),

dict(id="IC7", name="Controlled vocabulary: facility type has at most 16 distinct values",
     kind="domain constraint", u1="REQUIRED (U1 groups by facility type)",
     before="SELECT COUNT(DISTINCT COALESCE(facility_type,'<blank>')) FROM raw",
     after ="SELECT COUNT(DISTINCT facility_group) FROM insp",
     unit="distinct values"),

dict(id="IC8", name="Coverage: every non-blank facility type maps to a real group (not 'Other')",
     kind="completeness", u1="REQUIRED (U1 groups by facility type)",
     before=NA, before_note="no grouping existed in D",
     after ="SELECT COUNT(*) FROM insp WHERE facility_group = 'Other'"),

dict(id="IC9", name="1NF: violation facts are atomic (no multi-valued violation cell)",
     kind="normal-form / denial constraint", u1="REQUIRED (U1 counts violation categories)",
     before="SELECT COUNT(*) FROM raw WHERE violations IS NOT NULL",
     after ="SELECT COUNT(*) FROM viol WHERE violation_title LIKE '%|%'"),

dict(id="IC10", name="Referential integrity: viol.inspection_id references insp.inspection_id",
     kind="inclusion dependency", u1="REQUIRED (U1 joins the two relations)",
     before=NA, before_note="no violation relation existed in D",
     after ="SELECT COUNT(*) FROM viol v\n"
            "LEFT JOIN insp i USING (inspection_id)\n"
            "WHERE i.inspection_id IS NULL"),

dict(id="IC11", name="FD: (code_era, violation_number) -> violation_title",
     kind="functional dependency", u1="REQUIRED (stops the 2018 code change merging categories)",
     before=NA, before_note="violation number/title not extractable in D",
     after ="SELECT COUNT(*) FROM (SELECT code_era, violation_number FROM viol\n"
            "                      GROUP BY 1,2 HAVING COUNT(DISTINCT violation_title) > 1)"),

dict(id="IC11b", name="FD counter-example: violation_number alone -> violation_title",
     kind="functional dependency", u1="diagnostic: shows why code_era is needed",
     before=NA, before_note="violation number/title not extractable in D",
     after ="SELECT COUNT(*) FROM (SELECT violation_number FROM viol\n"
            "                      GROUP BY 1 HAVING COUNT(DISTINCT violation_title) > 1)",
     expect_nonzero=True),

dict(id="IC12", name="Domain: violation_number lies inside its era's code book",
     kind="domain constraint", u1="REQUIRED (guards the category vocabulary)",
     before=NA, before_note="violation number not extractable in D",
     after ="SELECT COUNT(*) FROM viol\n"
            "WHERE NOT ( (code_era = 'pre_2018_code' AND (violation_number BETWEEN 1 AND 45 OR violation_number = 70))\n"
            "         OR (code_era = '2018_code'     AND  violation_number BETWEEN 1 AND 64) )"),

dict(id="IC13", name="Every Fail carries violation evidence, or is explicitly flagged",
     kind="denial constraint", u1="REQUIRED (prevents false zeros in U1)",
     before="SELECT COUNT(*) FROM raw WHERE results = 'Fail' AND violations IS NULL",
     after ="SELECT COUNT(*) FROM insp i\n"
            "WHERE i.result = 'Fail'\n"
            "  AND NOT EXISTS (SELECT 1 FROM viol v WHERE v.inspection_id = i.inspection_id)\n"
            "  AND i.fail_missing_violations = FALSE"),

dict(id="IC14", name="Format: inspection_date is ISO-8601 and inside the published extract window",
     kind="format / range constraint", u1="REQUIRED (the date determines the code era)",
     before="SELECT COUNT(*) FROM raw\n"
            "WHERE inspection_date IS NULL\n"
            "   OR NOT REGEXP_MATCHES(inspection_date,'^[0-9]{2}/[0-9]{2}/[0-9]{4}$')",
     after ="SELECT COUNT(*) FROM insp\n"
            "WHERE inspection_date IS NULL\n"
            "   OR NOT REGEXP_MATCHES(inspection_date,'^[0-9]{4}-[0-9]{2}-[0-9]{2}$')\n"
            "   OR inspection_date < '2010-01-01' OR inspection_date > '2025-10-23'"),
]

def run_ic(sql):
    return None if sql is None else int(con.sql(sql).fetchone()[0])

_rows = []
for c in IC:
    b, a = run_ic(c.get("before")), run_ic(c.get("after"))
    if c.get("expect_nonzero"):
        verdict = "diagnostic"
    elif b is None:
        verdict = "new check, only possible on D-prime"
    elif c.get("unit") == "distinct values":
        verdict = "IMPROVED (%d -> %d)" % (b, a)
    elif a == 0 and b > 0:
        verdict = "RESOLVED"
    elif a == 0 and b == 0:
        verdict = "already clean (no change needed)"
    else:
        verdict = "partial (%d -> %d)" % (b, a)
    _rows.append({"IC": c["id"], "constraint": c["name"], "type": c["kind"],
                  "needed_for_U1": c["u1"], "unit": c.get("unit", "violating rows"),
                  "before_D": b if b is not None else "n/a - " + c.get("before_note", ""),
                  "after_Dprime": a, "verdict": verdict})
ic_report = pd.DataFrame(_rows)
ic_report

,IC,constraint,type,needed_for_U1,unit,before_D,after_Dprime,verdict
0,IC1,Key: inspection_id is unique,key constraint,required (entity identity),violating rows,0,0,already clean (no change needed)
1,IC2,"Domain: Risk in {Risk 1 (High), Risk 2 (Medium), Risk 3 (Low), Unknown}",domain constraint,REQUIRED (U1 groups by risk),violating rows,167,0,RESOLVED
2,IC3,Format: City is non-null and in canonical upper/trimmed form,format / completeness,useful (scope verification),violating rows,829,0,RESOLVED
3,IC4,Consistency: no misspelled single-token variant of CHICAGO survives,denial constraint,useful (scope verification),violating rows,96,0,RESOLVED
4,IC5,Domain: State is a 2-letter code or explicitly UNKNOWN,domain constraint,useful (scope verification),violating rows,58,0,RESOLVED
5,IC6,Domain: Results in the 7 documented outcomes (the U0 control),domain constraint,required (U1 filters Results='Fail'),violating rows,0,0,already clean (no change needed)
6,IC7,Controlled vocabulary: facility type has at most 16 distinct values,domain constraint,REQUIRED (U1 groups by facility type),distinct values,521,16,IMPROVED (521 -> 16)
7,IC8,Coverage: every non-blank facility type maps to a real group (not 'Other'),completeness,REQUIRED (U1 groups by facility type),violating rows,n/a - no grouping existed in D,190,"new check, only possible on D-prime"
8,IC9,1NF: violation facts are atomic (no multi-valued violation cell),normal-form / denial constraint,REQUIRED (U1 counts violation categories),violating rows,215525,0,RESOLVED
9,IC10,Referential integrity: viol.inspection_id references insp.inspection_id,inclusion dependency,REQUIRED (U1 joins the two relations),violating rows,n/a - no violation relation existed in D,0,"new check, only possible on D-prime"


In [27]:
_num = ic_report[pd.to_numeric(ic_report["before_D"], errors="coerce").notna()].copy()
_num["before_D"] = _num["before_D"].astype(int)
_rowunit = _num[_num.unit == "violating rows"]
ic_before_total = int(_rowunit.before_D.sum())
ic_after_total  = int(_rowunit.after_Dprime.sum())

print("Integrity-constraint summary")
print("  constraints checked                      :", len(ic_report))
print("  ... checkable on BOTH D and D-prime      :", len(_num))
print("  ... newly checkable only on D-prime      :",
      int(ic_report["before_D"].astype(str).str.startswith("n/a").sum()))
print("  IC-violating rows before cleaning        :", format(ic_before_total, ","))
print("  IC-violating rows after cleaning         :", format(ic_after_total, ","))
print("  constraints resolved (violations -> 0)   :", int((ic_report.verdict == "RESOLVED").sum()))
print("  constraints already satisfied in D (U0)  :",
      int(ic_report.verdict.str.startswith("already clean").sum()))

ic_report.to_csv(os.path.join(OUT_DIR, "ic_report.csv"), index=False)

log_op("7. IC checks", "run_integrity_constraints",
       "Ran %d integrity constraints as DuckDB SQL over D and over D-prime." % len(IC),
       "Supplies the measured evidence for Report 2b that the properties U1 depends on now hold.",
       params={"engine": "duckdb " + duckdb.__version__, "n_constraints": len(IC)},
       before={"ic_violating_rows": ic_before_total},
       after={"ic_violating_rows": ic_after_total},
       columns_out=["ic_report.csv"])

Integrity-constraint summary
  constraints checked                      : 15
  ... checkable on BOTH D and D-prime      : 10
  ... newly checkable only on D-prime      : 5
  IC-violating rows before cleaning        : 220,254
  IC-violating rows after cleaning         : 0
  constraints resolved (violations -> 0)   : 6
  constraints already satisfied in D (U0)  : 3
[op014] 7. IC checks :: run_integrity_constraints  |  before={'ic_violating_rows': 220254} -> after={'ic_violating_rows': 0}


---
# Step 8 — Data-change summary `ΔD`: which cells actually changed?

**What.** For every column with a raw counterpart, count (a) **cells whose value changed** and
(b) **cells that were blank and are now populated**, alongside distinct-value cardinality before and
after. Newly derived columns are listed separately, as is the structural 1→N change.

**Why for U1.** *Necessary as evidence.* Rubric §2a asks for *cells changed per column*, not
distinct-value counts, and the two tell very different stories: cleaning `City` touches only a few
hundred cells but removes 18 spurious categories, while `Inspection Date` rewrites **all 298,869**
cells without changing a single distinct day. Reporting both is what makes ΔD honest.

*(Latitude/Longitude are compared numerically rather than as text, so that a pure float-formatting
difference is not miscounted as a data change.)*

In [28]:
PAIRS = [
    ("City",            raw["City"],            clean_inspections["city_clean"],      "text"),
    ("State",           raw["State"],           clean_inspections["state_clean"],     "text"),
    ("Risk",            raw["Risk"],            clean_inspections["risk_clean"],      "text"),
    ("Facility Type",   raw["Facility Type"],   clean_inspections["facility_group"],  "text"),
    ("Results",         raw["Results"],         clean_inspections["result"],          "text"),
    ("Inspection Type", raw["Inspection Type"], clean_inspections["inspection_type"], "text"),
    ("Inspection Date", raw["Inspection Date"], clean_inspections["inspection_date"], "text"),
    ("DBA Name",        raw["DBA Name"],        clean_inspections["dba_name"],        "text"),
    ("AKA Name",        raw["AKA Name"],        clean_inspections["aka_name"],        "text"),
    ("Address",         raw["Address"],         clean_inspections["address"],         "text"),
    ("License #",       raw["License #"],       clean_inspections["license_num"],     "text"),
    ("Zip",             raw["Zip"],             clean_inspections["zip"],             "text"),
    ("Latitude",        raw["Latitude"],        clean_inspections["latitude"],        "numeric"),
    ("Longitude",       raw["Longitude"],       clean_inspections["longitude"],       "numeric"),
]

_rows = []
for name, b, a, mode in PAIRS:
    if mode == "numeric":
        bn = pd.to_numeric(b, errors="coerce")
        n_changed = int(((bn.notna() != a.notna()) | ((bn - a).abs() > 1e-9).fillna(False)).sum())
    else:
        n_changed = changed_count(b, a)
    _rows.append({
        "source_column":           name,
        "cleaned_column":          a.name,
        "cells_changed":           n_changed,
        "pct_cells_changed":       round(n_changed / len(raw) * 100, 3),
        "cells_filled_from_blank": filled_count(b, a),
        "distinct_before":         int(b.nunique(dropna=False)),
        "distinct_after":          int(a.nunique(dropna=False)),
        "missing_before":          int(b.isna().sum()),
        "missing_after":           int(a.isna().sum()),
    })
change_summary = pd.DataFrame(_rows).sort_values("cells_changed", ascending=False)
change_summary

,source_column,cleaned_column,cells_changed,pct_cells_changed,cells_filled_from_blank,distinct_before,distinct_after,missing_before,missing_after
6,Inspection Date,inspection_date,298869,100.000,0,3985,3985,0,0
5,Inspection Type,inspection_type,63561,21.267,0,111,101,1,1
3,Facility Type,facility_group,37459,12.534,5264,521,16,5264,0
7,DBA Name,dba_name,22607,7.564,0,34146,33634,0,0
8,AKA Name,aka_name,21991,7.358,0,32506,32000,2412,2412
9,Address,address,17887,5.985,0,20208,20038,3,3
0,City,city_clean,959,0.321,162,90,73,162,0
2,Risk,risk_clean,167,0.056,87,5,4,87,0
1,State,state_clean,58,0.019,58,7,7,58,0
4,Results,result,0,0.000,0,7,7,0,0


In [29]:
derived = pd.DataFrame([
    ("facility_group",          "derived", "16-value controlled vocabulary replacing 520 raw strings", len(raw)),
    ("risk_level",              "derived", "ordinal 1/2/3 (null for Unknown)", int(risk_level.notna().sum())),
    ("inspection_year",         "derived", "year of inspection", len(raw)),
    ("code_era",                "derived", "pre_2018_code / 2018_code (which violation code book)", len(raw)),
    ("is_reinspection",         "derived", "inspection type contains 're-inspection'", int(is_reinspection.sum())),
    ("has_violation_text",      "flag",    "inspection carried violation text", int(has_violation_text.sum())),
    ("n_violations",            "derived", "count of parsed violation rows", int((n_violations > 0).sum())),
    ("fail_missing_violations", "flag",    "Fail with no violation record", int(fail_missing_violations.sum())),
    ("risk_is_imputed",         "flag",    "risk was blank or 'All'", int(risk_is_imputed.sum())),
    ("facility_type_missing",   "flag",    "facility type blank", int(raw["Facility Type"].isna().sum())),
    ("missing_city",            "flag",    "city blank or a placeholder", int(flags.missing_city.sum())),
    ("missing_state",           "flag",    "state blank", int(flags.missing_state.sum())),
    ("out_of_state",            "flag",    "state present and not IL", int(out_of_state.sum())),
    ("missing_zip",             "flag",    "zip blank", int(flags.missing_zip.sum())),
    ("missing_geo",             "flag",    "latitude/longitude blank", int(flags.missing_geo.sum())),
], columns=["column", "kind", "meaning", "rows_true_or_populated"])
print("NEW columns in D-prime (none of these existed in D):")
print(derived.to_string(index=False))

print("\n\nSTRUCTURAL change - the 1:N decomposition:")
struct = pd.DataFrame([
    ("relations", 1, 2),
    ("inspection rows", len(raw), len(clean_inspections)),
    ("violation rows (atomic facts)", 0, len(clean_violations)),
    ("columns on the inspection relation", raw.shape[1], clean_inspections.shape[1]),
    ("queryable violation categories", 0, int(clean_violations.violation_title.nunique())),
], columns=["measure", "D", "D_prime"])
print(struct.to_string(index=False))

change_summary.to_csv(os.path.join(OUT_DIR, "change_summary.csv"), index=False)
derived.to_csv(os.path.join(OUT_DIR, "derived_columns.csv"), index=False)

total_changed = int(change_summary.cells_changed.sum())
total_cells   = len(raw) * raw.shape[1]
print("\nTOTAL cells rewritten in D: %s  (%.2f%% of the %s cells in D)"
      % (format(total_changed, ","), total_changed / total_cells * 100, format(total_cells, ",")))

log_op("8. Change summary", "quantify_cell_level_changes",
       "Diffed every cleaned column against its raw counterpart to produce per-column cells-changed "
       "and cells-filled counts (Report 2a).",
       "Quantifies delta-D: exactly which cells the workflow touched, so the cleaning is auditable.",
       before={"total_cells": total_cells},
       after={"cells_rewritten": total_changed, "new_columns": len(derived),
              "new_violation_rows": len(clean_violations)},
       rows_affected=total_changed,
       columns_out=["change_summary.csv", "derived_columns.csv"])

NEW columns in D-prime (none of these existed in D):
                 column    kind                                                  meaning  rows_true_or_populated
         facility_group derived 16-value controlled vocabulary replacing 520 raw strings                  298869
             risk_level derived                         ordinal 1/2/3 (null for Unknown)                  298702
        inspection_year derived                                       year of inspection                  298869
               code_era derived    pre_2018_code / 2018_code (which violation code book)                  298869
        is_reinspection derived                 inspection type contains 're-inspection'                   57957
     has_violation_text    flag                        inspection carried violation text                  215525
           n_violations derived                           count of parsed violation rows                  215525
fail_missing_violations    flag            

---
# Step 9 — The payoff: answering **U1** on `D` versus on `D'`

**What.** Run the U1 question — *most common violation categories in failed inspections, by facility
type and risk level* — first against the raw table, then against the cleaned relations.

**Counting rule.** A category counts **once per inspection** (`COUNT(DISTINCT inspection_id)`),
implemented in the query rather than by deleting the 59,359 redundant repeated-code rows from
`clean_violations`. Rows flagged `fail_missing_violations` cannot contribute (they have no violation
rows) and are reported separately, so the denominator is explicit rather than implied.

In [30]:
# ---------------------------------------------------------------- U1 attempted on RAW D
u1_raw_groups = con.sql(
    "SELECT COUNT(*) FROM (SELECT facility_type, risk, violations FROM raw "
    "WHERE results = 'Fail' GROUP BY 1,2,3)").fetchone()[0]
n_fail_raw = con.sql("SELECT COUNT(*) FROM raw WHERE results = 'Fail'").fetchone()[0]

print("U1 attempted directly on D   (GROUP BY facility_type, risk, violations):")
print("  failed inspections    : %s" % format(n_fail_raw, ","))
print("  resulting 'categories': %s" % format(u1_raw_groups, ","))
print("  -> %.1f%% of failed inspections form their own singleton group." % (u1_raw_groups / n_fail_raw * 100))
print("  The grouping key is the entire free-text blob, so the query returns roughly one row per")
print("  inspection. U1 is NOT ANSWERABLE on D by any GROUP BY.\n")

print("Best available raw work-around - substring search for one hand-picked category:")
print(con.sql("SELECT COUNT(*) AS inspections_matching_LIKE_temperature FROM raw "
              "WHERE results = 'Fail' AND violations ILIKE '%TEMPERATURE%'").df().to_string(index=False))
print("  ...but this requires the analyst to already know the category name, matches inspector")
print("  COMMENTS as well as titles, cannot be enumerated, and cannot be ranked.")
print("  That is text search, not analysis.")

U1 attempted directly on D   (GROUP BY facility_type, risk, violations):
  failed inspections    : 57,819
  resulting 'categories': 54,281
  -> 93.9% of failed inspections form their own singleton group.
  The grouping key is the entire free-text blob, so the query returns roughly one row per
  inspection. U1 is NOT ANSWERABLE on D by any GROUP BY.

Best available raw work-around - substring search for one hand-picked category:
 inspections_matching_LIKE_temperature
                                 16534
  ...but this requires the analyst to already know the category name, matches inspector
  COMMENTS as well as titles, cannot be enumerated, and cannot be ranked.
  That is text search, not analysis.


In [31]:
# ---------------------------------------------------------------- U1 answered on D-prime
U1_SQL = (
"SELECT i.facility_group,\n"
"       i.risk_clean,\n"
"       v.code_era,\n"
"       v.violation_number,\n"
"       v.violation_title,\n"
"       COUNT(DISTINCT i.inspection_id) AS n_failed_inspections\n"
"FROM insp i\n"
"JOIN viol v USING (inspection_id)\n"
"WHERE i.result = 'Fail'\n"
"  AND i.fail_missing_violations = FALSE\n"
"  AND v.violation_number_valid\n"
"GROUP BY 1,2,3,4,5\n"
"ORDER BY n_failed_inspections DESC")

u1 = con.sql(U1_SQL).df()
u1.to_csv(os.path.join(OUT_DIR, "u1_result.csv"), index=False)
print("U1 answered on D-prime: %s result cells over %d distinct categories\n"
      % (format(len(u1), ","), u1.violation_title.nunique()))
print("TOP 15 (facility_group x risk x violation category) among FAILED inspections:")
print(u1.head(15).to_string(index=False))

U1 answered on D-prime: 2,296 result cells over 109 distinct categories

TOP 15 (facility_group x risk x violation category) among FAILED inspections:
facility_group    risk_clean      code_era  violation_number                                                                                                                        violation_title  n_failed_inspections
    Restaurant Risk 1 (High) pre_2018_code                34                                  FLOORS: CONSTRUCTED PER CODE, CLEANED, GOOD REPAIR, COVING INSTALLED, DUST-LESS CLEANING METHODS USED                 10264
    Restaurant Risk 1 (High) pre_2018_code                35                   WALLS, CEILINGS, ATTACHED EQUIPMENT CONSTRUCTED PER CODE: GOOD REPAIR, SURFACES CLEAN AND DUST-LESS CLEANING METHODS                  9590
    Restaurant Risk 1 (High) pre_2018_code                33                                                        FOOD AND NON-FOOD CONTACT EQUIPMENT UTENSILS CLEAN, FREE OF ABRASIVE DETERGENTS

In [32]:
# ---------------------------------------------------------------- headline U1 views
print("Most common violation categories in failed inspections, overall")
print("(each category counted once per inspection):\n")
print(con.sql(
"SELECT v.code_era, v.violation_number, v.violation_title, "
"       COUNT(DISTINCT i.inspection_id) AS n_failed_inspections "
"FROM insp i JOIN viol v USING (inspection_id) "
"WHERE i.result = 'Fail' AND i.fail_missing_violations = FALSE "
"GROUP BY 1,2,3 ORDER BY 4 DESC LIMIT 12").df().to_string(index=False))

print("\n\nTop-2 violation categories per facility group  (2018 code era, Risk 1 (High) only):\n")
print(con.sql(
"WITH c AS ( "
"  SELECT i.facility_group, v.violation_title, COUNT(DISTINCT i.inspection_id) AS n "
"  FROM insp i JOIN viol v USING (inspection_id) "
"  WHERE i.result = 'Fail' AND i.fail_missing_violations = FALSE "
"    AND i.risk_clean = 'Risk 1 (High)' AND v.code_era = '2018_code' "
"  GROUP BY 1,2) "
"SELECT facility_group, violation_title, n "
"FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY facility_group ORDER BY n DESC) rk FROM c) "
"WHERE rk <= 2 ORDER BY facility_group, n DESC").df().to_string(index=False))

Most common violation categories in failed inspections, overall
(each category counted once per inspection):

     code_era  violation_number                                                                                                                        violation_title  n_failed_inspections
pre_2018_code                34                                  FLOORS: CONSTRUCTED PER CODE, CLEANED, GOOD REPAIR, COVING INSTALLED, DUST-LESS CLEANING METHODS USED                 19045
pre_2018_code                35                   WALLS, CEILINGS, ATTACHED EQUIPMENT CONSTRUCTED PER CODE: GOOD REPAIR, SURFACES CLEAN AND DUST-LESS CLEANING METHODS                 17854
pre_2018_code                18 NO EVIDENCE OF RODENT OR INSECT OUTER OPENINGS PROTECTED/RODENT PROOFED, A WRITTEN LOG SHALL BE MAINTAINED AVAILABLE TO THE INSPECTORS                 16413
pre_2018_code                33                                                        FOOD AND NON-FOOD CONTACT EQUIPMENT UTENSILS CL

In [33]:
# ---------------------------------------------------------------- coverage of the U1 grid
print("Failed-inspection coverage of the U1 grid (which rows actually contribute):\n")
print(con.sql(
"SELECT risk_clean, "
"       COUNT(*) FILTER (WHERE result='Fail')                                 AS failed_inspections, "
"       COUNT(*) FILTER (WHERE result='Fail' AND fail_missing_violations)     AS excluded_no_evidence, "
"       COUNT(*) FILTER (WHERE result='Fail' AND NOT fail_missing_violations) AS usable_for_U1 "
"FROM insp GROUP BY 1 ORDER BY 1").df().to_string(index=False))

_n_fail    = int((clean_inspections.result == "Fail").sum())
_n_imputed = int(((clean_inspections.result == "Fail") & clean_inspections.risk_is_imputed).sum())
print("\nWhat the 'All'/blank risk decision costs U1: %d failed inspections sit in the Unknown risk "
      "bucket (%.3f%% of failures)." % (_n_imputed, _n_imputed / max(_n_fail, 1) * 100))

log_op("9. U1", "answer_use_case_U1",
       "Ran the U1 analysis query on D (not answerable) and on D-prime, counting each violation "
       "category once per inspection via COUNT(DISTINCT inspection_id).",
       "Demonstrates fitness for use: the cleaning is sufficient for U1 and each step was necessary.",
       params={"sql": " ".join(U1_SQL.split())},
       before={"answerable": False, "groups_returned_on_D": int(u1_raw_groups)},
       after={"answerable": True, "result_cells": len(u1),
              "distinct_categories": int(u1.violation_title.nunique())},
       columns_out=["u1_result.csv"])

Failed-inspection coverage of the U1 grid (which rows actually contribute):

     risk_clean  failed_inspections  excluded_no_evidence  usable_for_U1
  Risk 1 (High)               42534                  2089          40445
Risk 2 (Medium)               10291                   665           9626
   Risk 3 (Low)                4968                   800           4168
        Unknown                  26                    25              1

What the 'All'/blank risk decision costs U1: 26 failed inspections sit in the Unknown risk bucket (0.045% of failures).
[op016] 9. U1 :: answer_use_case_U1  |  before={'answerable': False, 'groups_returned_on_D': 54281} -> after={'answerable': True, 'result_cells': 2296, 'distinct_categories': 109}


---
# Step 10 — Export the supplementary artifacts

**What.** Write `queries.txt` (every SQL statement executed, with its before/after result recorded
next to it) and `OtherToolHistory.json` (the machine-readable operation history that stands in for
`OpenRefineHistory.json`), then print a manifest of everything produced.

**Why.** Rubric §5: the workflow has to be reproducible and inspectable by someone who never runs
this notebook.

In [34]:
def fmt_sql(sql):
    # left-align a SQL block that was written as an indented python string
    ls = [l.rstrip() for l in sql.strip("\n").split("\n")]
    tail = [l for l in ls[1:] if l.strip()]
    pad = min([len(l) - len(l.lstrip()) for l in tail], default=0)
    return "\n".join([ls[0]] + [(l[pad:] if l.strip() else "") for l in ls[1:]])

L = []
L.append("=" * 100)
L.append("CS513 Phase-II  -  queries.txt")
L.append("Team79  -  Chicago Food Inspections")
L.append("Sid Wanjara (swanj2), Drew Patel (drewp4), Shray Srivastava (ssriv5)")
L.append("")
L.append("Engine: DuckDB %s, executed in-process from CS513_Phase2_Cleaning.ipynb" % duckdb.__version__)
L.append("Relations:")
L.append("  raw  = Food-Inspections-20251023.csv loaded verbatim, columns snake_cased        (D)")
L.append("  insp = output/clean_inspections.csv  - one row per inspection                    (D-prime)")
L.append("  viol = output/clean_violations.csv   - one row per violation per inspection      (D-prime)")
L.append("")
L.append("Every IC query returns the NUMBER OF VIOLATING ROWS; 0 means the constraint holds.")
L.append("The results recorded below were produced by the notebook run started %s." % RUN_STARTED)
L.append("=" * 100)
L.append("")
L.append("#" * 100)
L.append("# PART 1  -  INTEGRITY CONSTRAINTS   (before = evaluated on D, after = evaluated on D-prime)")
L.append("#" * 100)
for c, r in zip(IC, ic_report.to_dict("records")):
    L.append("")
    L.append("-" * 100)
    L.append("-- %s  %s" % (c["id"], c["name"]))
    L.append("--    type          : %s" % c["kind"])
    L.append("--    needed for U1 : %s" % c["u1"])
    L.append("--    RESULT        : before(D) = %s   after(D-prime) = %s   [%s]"
             % (r["before_D"], r["after_Dprime"], r["verdict"]))
    L.append("-" * 100)
    if c.get("before"):
        L.append("-- %s BEFORE (on D):" % c["id"])
        L.append(fmt_sql(c["before"]) + ";")
    else:
        L.append("-- %s BEFORE: not expressible on D - %s" % (c["id"], c.get("before_note", "")))
    L.append("")
    L.append("-- %s AFTER (on D-prime):" % c["id"])
    L.append(fmt_sql(c["after"]) + ";")

L.append("")
L.append("#" * 100)
L.append("# PART 2  -  USE CASE U1")
L.append("#" * 100)
L.append("")
L.append("-- U1 attempted on the RAW data. The only available grouping key is the whole free-text")
L.append("-- Violations blob, which yields %s groups for %s failed inspections:"
         % (format(u1_raw_groups, ","), format(n_fail_raw, ",")))
L.append("SELECT facility_type, risk, violations, COUNT(*) AS n")
L.append("FROM raw WHERE results = 'Fail' GROUP BY 1,2,3 ORDER BY n DESC;")
L.append("")
L.append("-- U1 answered on the CLEANED data (each category counted once per inspection):")
L.append(U1_SQL + ";")
L.append("")
L.append("-- U1, headline view: most common categories among failed inspections, overall")
L.append("SELECT v.code_era, v.violation_number, v.violation_title,")
L.append("       COUNT(DISTINCT i.inspection_id) AS n_failed_inspections")
L.append("FROM insp i JOIN viol v USING (inspection_id)")
L.append("WHERE i.result = 'Fail' AND i.fail_missing_violations = FALSE")
L.append("GROUP BY 1,2,3 ORDER BY 4 DESC LIMIT 12;")
L.append("")
L.append("-- U1, top-2 categories per facility group, Risk 1 (High), 2018 code era")
L.append("WITH c AS (")
L.append("  SELECT i.facility_group, v.violation_title, COUNT(DISTINCT i.inspection_id) AS n")
L.append("  FROM insp i JOIN viol v USING (inspection_id)")
L.append("  WHERE i.result = 'Fail' AND i.fail_missing_violations = FALSE")
L.append("    AND i.risk_clean = 'Risk 1 (High)' AND v.code_era = '2018_code'")
L.append("  GROUP BY 1,2)")
L.append("SELECT facility_group, violation_title, n")
L.append("FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY facility_group ORDER BY n DESC) rk FROM c)")
L.append("WHERE rk <= 2 ORDER BY facility_group, n DESC;")

L.append("")
L.append("#" * 100)
L.append("# PART 3  -  PROFILING QUERIES  (Step 1; run as pandas in the notebook, SQL equivalents here)")
L.append("#" * 100)
L.append("")
L.append("-- missing values per column, on D")
L.append("SELECT 'Violations'    AS col, COUNT(*) FILTER (WHERE violations    IS NULL) AS n_missing FROM raw")
L.append("UNION ALL SELECT 'Facility Type', COUNT(*) FILTER (WHERE facility_type IS NULL) FROM raw")
L.append("UNION ALL SELECT 'AKA Name',      COUNT(*) FILTER (WHERE aka_name      IS NULL) FROM raw")
L.append("UNION ALL SELECT 'City',          COUNT(*) FILTER (WHERE city          IS NULL) FROM raw")
L.append("UNION ALL SELECT 'Risk',          COUNT(*) FILTER (WHERE risk          IS NULL) FROM raw")
L.append("UNION ALL SELECT 'State',         COUNT(*) FILTER (WHERE state         IS NULL) FROM raw")
L.append("UNION ALL SELECT 'Zip',           COUNT(*) FILTER (WHERE zip           IS NULL) FROM raw")
L.append("UNION ALL SELECT 'Latitude',      COUNT(*) FILTER (WHERE latitude      IS NULL) FROM raw")
L.append("ORDER BY n_missing DESC;")
L.append("")
L.append("-- distribution of the U1 grouping dimensions, BEFORE cleaning")
L.append("SELECT risk,          COUNT(*) n FROM raw GROUP BY 1 ORDER BY n DESC;")
L.append("SELECT facility_type, COUNT(*) n FROM raw GROUP BY 1 ORDER BY n DESC LIMIT 25;")
L.append("SELECT city,          COUNT(*) n FROM raw GROUP BY 1 ORDER BY n DESC LIMIT 25;")
L.append("")
L.append("-- ... and AFTER cleaning")
L.append("SELECT risk_clean,     COUNT(*) n FROM insp GROUP BY 1 ORDER BY n DESC;")
L.append("SELECT facility_group, COUNT(*) n FROM insp GROUP BY 1 ORDER BY n DESC;")
L.append("SELECT city_clean,     COUNT(*) n FROM insp GROUP BY 1 ORDER BY n DESC LIMIT 25;")
L.append("")
L.append("-- violation-level profile (only possible on D-prime)")
L.append("SELECT code_era, COUNT(*) AS violation_rows, COUNT(DISTINCT violation_title) AS categories,")
L.append("       COUNT(DISTINCT inspection_id) AS inspections")
L.append("FROM viol GROUP BY 1;")
L.append("")
L.append("-- repeated violation codes inside a single inspection")
L.append("-- (kept in D-prime, de-duplicated at query time for U1)")
L.append("SELECT COUNT(*) FROM (SELECT inspection_id, violation_number FROM viol")
L.append("                      GROUP BY 1,2 HAVING COUNT(*) > 1);")
L.append("")

qpath = os.path.join(PROJECT_DIR, "queries.txt")
with open(qpath, "w") as f:
    f.write("\n".join(L))
print("wrote %s  (%.1f KB, %d lines)" % (qpath, os.path.getsize(qpath) / 1024, len(L)))

wrote /Users/shraysrivastava/Desktop/cs513/queries.txt  (15.8 KB, 259 lines)

In [35]:
history = {
    "project": "CS513 Phase-II - Chicago Food Inspections",
    "team": "Team79",
    "members": [
        {"name": "Sid Wanjara", "netid": "swanj2"},
        {"name": "Drew Patel", "netid": "drewp4"},
        {"name": "Shray Srivastava", "netid": "ssriv5"},
    ],
    "note": ("Operation history for our data-cleaning workflow. We did not use OpenRefine, so this "
             "file replaces OpenRefineHistory.json. Every record was emitted by "
             "CS513_Phase2_Cleaning.ipynb at run time, in execution order."),
    "tooling": {
        "cleaning": "Python %s / pandas %s" % (sys.version.split()[0], pd.__version__),
        "integrity_checks": "DuckDB %s (in-process SQL over the dataframes)" % duckdb.__version__,
        "notebook": "CS513_Phase2_Cleaning.ipynb",
    },
    "input": {"file": os.path.basename(RAW_CSV), "rows": int(len(raw)), "columns": int(raw.shape[1])},
    "outputs": {
        "clean_inspections.csv": {"rows": int(len(clean_inspections)),
                                  "columns": int(clean_inspections.shape[1]),
                                  "grain": "one row per inspection", "key": "inspection_id"},
        "clean_violations.csv":  {"rows": int(len(clean_violations)),
                                  "columns": int(clean_violations.shape[1]),
                                  "grain": "one row per violation per inspection",
                                  "key": "(inspection_id, violation_seq)",
                                  "foreign_key": "inspection_id -> clean_inspections.inspection_id"},
    },
    "run_started": RUN_STARTED,
    "run_finished": datetime.datetime.now().isoformat(timespec="seconds"),
    "n_operations": len(OPS),
    "operations": OPS,
    "integrity_constraints": ic_report.to_dict("records"),
    "change_summary": change_summary.to_dict("records"),
}
hpath = os.path.join(OUT_DIR, "OtherToolHistory.json")
with open(hpath, "w") as f:
    json.dump(history, f, indent=2, default=str)
print("wrote %s  (%.1f KB) - %d logged operations\n" % (hpath, os.path.getsize(hpath) / 1024, len(OPS)))
print(pd.DataFrame(OPS)[["op_id", "step", "operation", "rows_affected"]].to_string(index=False))

wrote /Users/shraysrivastava/Desktop/cs513/output/OtherToolHistory.json  (26.9 KB) - 16 logged operations

op_id                 step                                operation  rows_affected
op001           1. Profile                    verify_phase1_profile            NaN
op002           1. Profile                profile_violations_column            NaN
op003             2.1 City               normalize_and_cluster_city          959.0
op004            2.2 State                          normalize_state           58.0
op005             2.3 Risk               map_risk_to_ordinal_domain          167.0
op006          2.4 Results                    verify_results_domain            0.0
op007  2.5 Inspection Type                normalize_inspection_type        63561.0
op008  2.5 Inspection Date          parse_dates_and_derive_code_era       298869.0
op009 2.6 Identity/address                     trim_identity_fields        40494.0
op010     3. Facility type               map_facility_type_to_g

In [36]:
print("=" * 92)
print("RUN COMPLETE -", datetime.datetime.now().isoformat(timespec="seconds"))
print("=" * 92)
man = [(os.path.relpath(p, PROJECT_DIR), "%.2f MB" % (os.path.getsize(p) / 1e6))
       for p in [os.path.join(OUT_DIR, f) for f in sorted(os.listdir(OUT_DIR))] + [qpath]]
print(pd.DataFrame(man, columns=["artifact", "size"]).to_string(index=False))
print()
print("D       : %s rows x %d cols, 1 relation" % (format(len(raw), ","), raw.shape[1]))
print("D-prime : %s rows x %d cols (inspections) + %s rows x %d cols (violations), 2 relations"
      % (format(len(clean_inspections), ","), clean_inspections.shape[1],
         format(len(clean_violations), ","), clean_violations.shape[1]))
print("cells rewritten: %s | new columns: %d | violation facts recovered from free text: %s"
      % (format(total_changed, ","), len(derived), format(len(clean_violations), ",")))
print("IC-violating rows across comparable constraints: %s -> %s"
      % (format(ic_before_total, ","), format(ic_after_total, ",")))

RUN COMPLETE - 2026-07-27T22:10:36
                    artifact      size
output/OtherToolHistory.json   0.03 MB
   output/change_summary.csv   0.00 MB
output/clean_inspections.csv  91.69 MB
 output/clean_violations.csv 285.00 MB
  output/derived_columns.csv   0.00 MB
        output/ic_report.csv   0.00 MB
        output/u1_result.csv   0.26 MB
                 queries.txt   0.02 MB

D       : 298,869 rows x 17 cols, 1 relation
D-prime : 298,869 rows x 34 cols (inspections) + 974,597 rows x 8 cols (violations), 2 relations
cells rewritten: 463,558 | new columns: 15 | violation facts recovered from free text: 974,597
IC-violating rows across comparable constraints: 220,254 -> 0


---
# Step 11 — Findings, deviations from the Phase-I plan, and next steps

### What we found that the Phase-I plan did not anticipate

1. **The violation code book changed on 2018-07-01.** Phase-I recorded the valid violation numbers
   as *1–44 plus 70*; the data actually contains *1–64 plus 70*, because Chicago replaced its
   inspection checklist in July 2018. The same number denotes different violations on either side of
   that date (`3.` = *food temperature* before, *management/employee knowledge* after). Had we
   grouped U1 by violation number, **45 categories would each have silently merged two unrelated
   violations**. Fix: derive `code_era` from the inspection date, validate numbers against the
   era-specific domain, and use the *title* — which is in bijection with `(era, number)`, 110
   categories to 110 titles — as the category key.
2. **Repeated violation numbers within one inspection are real, not duplicates.** 59,359 rows across
   33,317 inspections repeat a code with *different inspector comments*. We keep them all and
   de-duplicate at query time with `COUNT(DISTINCT inspection_id)`.
3. **3,579 failed inspections carry no violation text at all** (6.2% of all failures). This is
   missing data that would otherwise read as "a failure with zero violations" and deflate every
   category rate in U1.
4. **Fuzzy clustering can move data in the wrong direction.** Frequency-based clustering of city
   names would have mapped the correctly spelled `MERRILLVILLE` into the misspelled `MERRIVILLE`.
   We restricted fuzzy merging to the single anchor value U1's scope depends on (`CHICAGO`).
5. **Not every DQ problem is worth fixing.** `Results` needed nothing (U0), and `DBA Name` entity
   resolution — the most *visible* remaining mess — is irrelevant to U1 and was deliberately skipped.

### Deviations from the Phase-I plan

| Phase-I plan | What we actually did | Why |
|---|---|---|
| Clean in OpenRefine, export `OpenRefineHistory.json` | Cleaned in Python/pandas; emitted `OtherToolHistory.json` from an in-notebook operation log | The 1→N decomposition of `Violations` into a second relation, plus the date-dependent code-book logic, are not expressible as an OpenRefine recipe |
| Visualize the inner workflow with OR2YW | `W2` drawn from this notebook's numbered step structure | OR2YW requires an OpenRefine history |
| Violation numbers 1–44 ∪ {70} | 1–45 ∪ {70} (pre-2018) and 1–64 (2018 code) | Measured from the data — see finding 1 |
| Owner of cleaning: Drew (S3) | Owner: Shray | Re-assigned during Phase-II planning |
| Fold city variants **and** suburb typos | Chicago variants only | Wrong-direction merge risk — see finding 4; suburbs are out of U1's scope |
| `Risk = 'All'` semantics unresolved | Mapped to `Unknown`, flagged via `risk_is_imputed` | 80 rows (0.03%), not an ordinal level. Recorded as an assumption, not a fact |

### Limitations of `D'`

* The `Other` facility group retains 190 rows (0.06%) that no rule could classify (`HERBALIFE`,
  `REGULATED BUSINESS`, `blockbuster video`, `Laundromat`, …); several of these are arguably not
  food establishments at all.
* 5,264 inspections have no facility type and sit in `Unknown`; they are not imputable from any
  other column without guessing.
* No entity resolution over `DBA Name` / `License #`, so `D'` cannot answer "how did *this chain*
  perform over time" — deliberately outside U1's scope.
* `D'` inherits `D`'s coverage: inspections, not health outcomes. **U2 remains unanswerable** by
  construction, exactly as predicted in Phase-I.

### How U1 would be implemented in production

Load the two relations into a small warehouse — the same DuckDB engine used here is enough —
materialize the Step 9 query as a view, and put a three-filter UI on it (facility group × risk level
× code era). The one non-obvious requirement to carry forward is the **code era**: any dashboard
that lets a user select a date range spanning 2018-07-01 must group by *title*, never by *number*, or
it will merge categories. The `fail_missing_violations` count belongs on the dashboard as a
data-quality footnote rather than hidden inside the pipeline.